In [ ]:
"""
Step 1: Fetch hourly solar/weather data for a Bangladesh target site from NASA POWER.

WHY: No public metered PV generation dataset exists for Bangladesh microgrids.
NASA POWER gives free, no-API-key, satellite-derived irradiance + weather for ANY
coordinates on Earth, hourly, back to 1981. We'll feed this into PVLIB in Step 3
to physically SIMULATE what a PV system would have produced there.

RUN THIS ON YOUR OWN MACHINE OR GOOGLE COLAB (not inside a locked-down sandbox) --
it needs to reach power.larc.nasa.gov directly.

USAGE:
    pip install requests pandas
    python 01_fetch_nasa_power.py
"""

import requests
import pandas as pd

# ---- CONFIGURE YOUR TARGET SITE ----
# Pick coordinates for the rural/microgrid region you want to represent.
# Dhaka is used as a default; swap in coordinates for a specific upazila if you
# want a more "rural microgrid" framing (e.g. somewhere in Rangpur, Khulna, or
# wherever your SDG/rural-electrification narrative is strongest).
LATITUDE = 23.8103
LONGITUDE = 90.4125
START_DATE = "20230101"   # YYYYMMDD
END_DATE = "20231231"     # one full year captures the monsoon/dry-season cycle

# Parameters: GHI (solar resource), air temp, humidity, wind speed, surface pressure.
# Max 20 parameters per request; we only need these 5 for PV simulation.
PARAMETERS = "ALLSKY_SFC_SW_DWN,T2M,RH2M,WS10M,PS"

BASE_URL = "https://power.larc.nasa.gov/api/temporal/hourly/point"


def fetch_nasa_power(lat, lon, start, end, parameters):
    params = {
        "parameters": parameters,
        "community": "RE",          # Renewable Energy community = solar-tuned dataset
        "longitude": lon,
        "latitude": lat,
        "start": start,
        "end": end,
        "format": "JSON",
        "time-standard": "UTC",
    }
    resp = requests.get(BASE_URL, params=params, timeout=120)
    resp.raise_for_status()
    payload = resp.json()
    data = payload["properties"]["parameter"]

    # Each parameter is a dict of {"YYYYMMDDHH": value}. Combine into one DataFrame.
    df = pd.DataFrame(data)
    df.index = pd.to_datetime(df.index, format="%Y%m%d%H")
    df.index.name = "timestamp"

    rename_map = {
        "ALLSKY_SFC_SW_DWN": "ghi_wh_m2",   # hourly GHI, Wh/m^2 (numerically = avg W/m^2 over the hour)
        "T2M": "temp_air_c",
        "RH2M": "rh_pct",
        "WS10M": "wind_speed_ms",
        "PS": "pressure_kpa",
    }
    df = df.rename(columns=rename_map)

    # NASA POWER uses -999 as a missing-value sentinel -- convert to NaN.
    df = df.replace(-999.0, pd.NA)
    return df


if __name__ == "__main__":
    df = fetch_nasa_power(LATITUDE, LONGITUDE, START_DATE, END_DATE, PARAMETERS)
    out_path = "bangladesh_target_weather_raw.csv"
    df.to_csv(out_path)
    print(f"Saved {len(df)} hourly rows to {out_path}")
    print(df.head())
    print("\nMissing values per column:")
    print(df.isna().sum())

Saved 8760 hourly rows to bangladesh_target_weather_raw.csv
                     ghi_wh_m2  temp_air_c  rh_pct  wind_speed_ms  \
timestamp                                                           
2023-01-01 00:00:00       0.00       11.38   98.60           2.02   
2023-01-01 01:00:00      51.62       12.76   93.40           1.74   
2023-01-01 02:00:00     161.85       14.94   83.31           1.98   
2023-01-01 03:00:00     290.02       17.59   72.23           1.71   
2023-01-01 04:00:00     406.60       20.59   57.82           2.03   

                     pressure_kpa  
timestamp                          
2023-01-01 00:00:00        101.83  
2023-01-01 01:00:00        101.92  
2023-01-01 02:00:00        101.98  
2023-01-01 03:00:00        102.01  
2023-01-01 04:00:00        101.99  

Missing values per column:
ghi_wh_m2        0
temp_air_c       0
rh_pct           0
wind_speed_ms    0
pressure_kpa     0
dtype: int64


In [ ]:
"""
Step 2: Load and clean a DKASC (Alice Springs) source-domain CSV.

WHY: DKASC is the standard open benchmark for transfer-learning PV forecasting
papers (climate: arid/tropical desert, opposite of Bangladesh's monsoon climate --
which is exactly the point: we pretrain on this data-rich, climatically different
source, then fine-tune on the data-scarce Bangladesh target).

HOW TO GET THE FILE (manual step, ~2 minutes):
    1. Go to https://dkasolarcentre.com.au/download?location=alice-springs
    2. Pick ONE site (recommend a mid-size, long-running one, e.g. "Site 1B" or
       "Site 25") and download its CSV (you may need to tick a date range).
    3. Save it as: dkasc_raw.csv  (in the same folder as this script)
    Note: the DKASC site has had intermittent maintenance outages in the past --
    if it's down, an alternative is to email info@dkasolarcentre.com.au, or
    swap in any other open PV time-series dataset with power + irradiance +
    temperature columns (e.g. an IEEE DataPort / Kaggle India PV dataset) --
    just adjust COLUMN_HINTS below to match its headers.

This script does NOT hardcode exact DKASC column names (their CSV export
schema has changed over the years). Instead it fuzzy-matches column names so
it keeps working even if the export format shifts slightly. ALWAYS print
df.columns after loading and sanity-check the auto-detected mapping below
before trusting it.

USAGE:
    pip install pandas numpy
    python 02_load_clean_dkasc.py
"""

import pandas as pd
import numpy as np

RAW_PATH = "/kaggle/input/datasets/rakibabdullah47/dkasc-raw/106-Site_DKA-M5_C-Phase.csv"
OUT_PATH = "/kaggle/working/dkasc_clean_hourly.csv"

# Substrings used to auto-detect each needed column, in priority order.
COLUMN_HINTS = {
    "timestamp":   ["timestamp", "date", "time"],
    "ac_power_kw": ["active_power", "ac_power", "power"],
    "ghi":         ["global_horizontal_radiation", "ghi", "global_horizontal"],
    "dhi":         ["diffuse_horizontal_radiation", "dhi", "diffuse_horizontal"],
    "temp_air_c":  ["weather_temperature", "temperature", "temp_air", "ambient_temp"],
    "wind_speed_ms": ["wind_speed"],
}

# Physically plausible ranges for Alice Springs. Anything outside these is a
# sensor fault / sentinel code, NOT a real reading -- convert to NaN before
# any interpolation, or it will silently corrupt the cell-temperature model
# downstream (wind speed and temperature both feed into PVLIB's SAPM model).
PHYSICAL_BOUNDS = {
    "ac_power_kw":   (-1, 50),      # allow small negative standby/parasitic noise; clip(lower=0) handles it below
    "ghi":           (0, 1400),     # solar constant ~1361 W/m^2; allow brief cloud-enhancement spikes
    "dhi":           (0, 1400),
    "temp_air_c":    (-10, 50),     # Alice Springs record low ~-7C, record high ~45C
    "wind_speed_ms": (0, 40),       # 40 m/s = 144 km/h, already an extreme storm
}


def find_column(columns, hints):
    cols_lower = {c.lower(): c for c in columns}
    for hint in hints:
        for lower_name, original_name in cols_lower.items():
            if hint in lower_name:
                return original_name
    return None


def load_and_clean(path):
    df_raw = pd.read_csv(path)
    print("Detected raw columns:", list(df_raw.columns))

    mapping = {}
    for target, hints in COLUMN_HINTS.items():
        found = find_column(df_raw.columns, hints)
        mapping[target] = found
        status = found if found else "NOT FOUND -- check COLUMN_HINTS / raw file"
        print(f"  {target:15s} <- {status}")

    if mapping["timestamp"] is None or mapping["ac_power_kw"] is None:
        raise ValueError(
            "Could not auto-detect 'timestamp' or 'power' column. "
            "Open the raw CSV, check the real header names, and update "
            "COLUMN_HINTS at the top of this script."
        )

    keep = {v: k for k, v in mapping.items() if v is not None}
    df = df_raw[list(keep.keys())].rename(columns=keep)

    df["timestamp"] = pd.to_datetime(df["timestamp"], errors="coerce")
    df = df.dropna(subset=["timestamp"]).set_index("timestamp").sort_index()

    # Coerce everything to numeric; non-numeric junk becomes NaN.
    for col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    # Sensor faults / sentinel codes (e.g. wind_speed = -142, temp = -40C at
    # a site that's never recorded below -7C) are NOT missing values in the
    # raw file -- they're present, numeric, and wrong. Null them out BEFORE
    # interpolation, or they will quietly poison the cell-temperature model
    # and any statistic computed downstream.
    print("\nOut-of-physical-range values found (converted to NaN):")
    for col, (low, high) in PHYSICAL_BOUNDS.items():
        if col in df.columns:
            bad = ~df[col].between(low, high) & df[col].notna()
            print(f"  {col:15s}: {bad.sum()} values outside [{low}, {high}]")
            df.loc[bad, col] = np.nan

    # Physically impossible values: power can't be negative (clip small
    # negative noise from inverter standby draw, common in raw DKASC data).
    df["ac_power_kw"] = df["ac_power_kw"].clip(lower=0)

    # DKASC is natively 5-minute resolution; resample to hourly means to match
    # the NASA POWER target-domain resolution used elsewhere in this pipeline.
    numeric_cols = df.select_dtypes(include=[np.number]).columns
    df_hourly = df[numeric_cols].resample("1h").mean()

    # Drop hours that are >50% missing within the resample window, then
    # linearly interpolate the small remaining gaps (do NOT interpolate large
    # gaps -- that would fabricate data; leave those as NaN and report them).
    df_hourly = df_hourly.interpolate(method="linear", limit=2)

    n_missing = df_hourly.isna().sum()
    print("\nRemaining missing values after light interpolation (limit=2h):")
    print(n_missing)

    return df_hourly


if __name__ == "__main__":
    df_clean = load_and_clean(RAW_PATH)
    df_clean.to_csv(OUT_PATH)
    print(f"\nSaved {len(df_clean)} cleaned hourly rows to {OUT_PATH}")
    print(df_clean.describe())

Detected raw columns: ['timestamp', 'Active_Energy_Delivered_Received', 'Current_Phase_Average', 'Active_Power', 'Performance_Ratio', 'Wind_Speed', 'Weather_Temperature_Celsius', 'Weather_Relative_Humidity', 'Global_Horizontal_Radiation', 'Diffuse_Horizontal_Radiation', 'Wind_Direction', 'Weather_Daily_Rainfall', 'Radiation_Global_Tilted', 'Radiation_Diffuse_Tilted']
  timestamp       <- timestamp
  ac_power_kw     <- Active_Power
  ghi             <- Global_Horizontal_Radiation
  dhi             <- Diffuse_Horizontal_Radiation
  temp_air_c      <- Weather_Temperature_Celsius
  wind_speed_ms   <- Wind_Speed

Out-of-physical-range values found (converted to NaN):
  ac_power_kw    : 0 values outside [-1, 50]
  ghi            : 69 values outside [0, 1400]
  dhi            : 2 values outside [0, 1400]
  temp_air_c     : 3051 values outside [-10, 50]
  wind_speed_ms  : 21 values outside [0, 40]

Remaining missing values after light interpolation (limit=2h):
ac_power_kw       1204
ghi       

In [ ]:
!pip install pvlib

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.4/19.4 MB 85.0 MB/s eta 0:00:00:00:0100:01


In [ ]:
"""
Step 3: Simulate the synthetic Bangladesh target PV power series from NASA
POWER weather data, using PVLIB's physical PV model.

WHY: This is the same strategy used by SolNet (Springer/arXiv, 2024-2025) when
no metered ground truth exists for a target region: take physically grounded
irradiance + weather, run it through a standard PV system model, and treat the
output as the target-domain "ground truth" for the transfer-learning + UQ
experiments. Document this choice explicitly in your paper's Methodology
section -- it's a legitimate, precedented substitute, not a shortcut to hide.

INPUT:  bangladesh_target_weather_raw.csv  (output of 01_fetch_nasa_power.py)
OUTPUT: bangladesh_target_pv_simulated.csv (synthetic AC power, hourly)

USAGE:
    pip install pvlib pandas numpy
    python 03_simulate_target_pv.py
"""

import pandas as pd
import numpy as np
import pvlib

IN_PATH = "/kaggle/working/bangladesh_target_weather_raw.csv"
OUT_PATH = "/kaggle/working/bangladesh_target_pv_simulated.csv"

LATITUDE = 23.8103
LONGITUDE = 90.4125
ALTITUDE = 8  # meters, approx for Dhaka region; adjust if you change coordinates

# A modest rooftop/microgrid-scale system -- comparable order of magnitude to
# a single DKASC site, which matters for a fair transfer-learning comparison.
SYSTEM_PDC0_KW = 10.0      # nameplate DC capacity
SURFACE_TILT = LATITUDE    # simple rule-of-thumb fixed tilt = latitude
SURFACE_AZIMUTH = 180      # south-facing (correct for northern hemisphere)
GAMMA_PDC = -0.0045        # %/degC power temperature coefficient, typical mono-Si
EFFICIENCY_INV = 0.96      # nominal inverter efficiency


def simulate(df):
    """df must have a DatetimeIndex (hourly, tz-naive local time) and a
    'ghi_wh_m2' column (numerically equal to average W/m^2 for that hour),
    plus 'temp_air_c' and 'wind_speed_ms'."""

    times = df.index
    solpos = pvlib.solarposition.get_solarposition(times, LATITUDE, LONGITUDE, altitude=ALTITUDE)

    ghi = df["ghi_wh_m2"].clip(lower=0)

    # Decompose GHI into DNI + DHI components (Erbs model) since NASA POWER
    # only gives us the combined GHI, but plane-of-array irradiance needs the
    # direct/diffuse split.
    erbs_out = pvlib.irradiance.erbs(ghi, solpos["zenith"], times)
    dni = erbs_out["dni"].fillna(0)
    dhi = erbs_out["dhi"].fillna(0)

    poa = pvlib.irradiance.get_total_irradiance(
        surface_tilt=SURFACE_TILT,
        surface_azimuth=SURFACE_AZIMUTH,
        solar_zenith=solpos["apparent_zenith"],
        solar_azimuth=solpos["azimuth"],
        dni=dni,
        ghi=ghi,
        dhi=dhi,
    )
    poa_global = poa["poa_global"].clip(lower=0)

    # Cell temperature via the SAPM open-rack model (reasonable default for a
    # rooftop/ground-mount microgrid system without manufacturer-specific data).
    temp_params = pvlib.temperature.TEMPERATURE_MODEL_PARAMETERS["sapm"]["open_rack_glass_glass"]
    temp_cell = pvlib.temperature.sapm_cell(
        poa_global=poa_global,
        temp_air=df["temp_air_c"],
        wind_speed=df["wind_speed_ms"].fillna(1.0),
        **temp_params,
    )

    pdc0_w = SYSTEM_PDC0_KW * 1000
    pdc = pvlib.pvsystem.pvwatts_dc(
        effective_irradiance=poa_global, temp_cell=temp_cell, pdc0=pdc0_w, gamma_pdc=GAMMA_PDC
    )
    pac = pvlib.inverter.pvwatts(pdc=pdc, pdc0=pdc0_w, eta_inv_nom=EFFICIENCY_INV)
    pac_kw = (pac / 1000).clip(lower=0)

    out = pd.DataFrame({
        "ac_power_kw": pac_kw,
        "poa_global_wm2": poa_global,
        "ghi_wm2": ghi,
        "temp_air_c": df["temp_air_c"],
        "wind_speed_ms": df["wind_speed_ms"],
    })
    return out


if __name__ == "__main__":
    df_in = pd.read_csv(IN_PATH, index_col=0, parse_dates=True)
    df_out = simulate(df_in)
    df_out.to_csv(OUT_PATH)
    print(f"Saved {len(df_out)} hourly simulated rows to {OUT_PATH}")
    print(df_out.describe())
    daily_kwh = df_out["ac_power_kw"].resample("1D").sum()
    print(f"\nMean simulated daily yield: {daily_kwh.mean():.2f} kWh/day "
          f"(~{daily_kwh.mean()/SYSTEM_PDC0_KW:.2f} kWh/kWp/day -- "
          f"sanity check: monsoon-belt Bangladesh should land roughly 3.5-4.5)")

Saved 8760 hourly simulated rows to /kaggle/working/bangladesh_target_pv_simulated.csv
       ac_power_kw  poa_global_wm2      ghi_wm2   temp_air_c  wind_speed_ms
count  8760.000000     8760.000000  8760.000000  8760.000000    8760.000000
mean      1.647122      190.248668   183.180258    25.827065       2.621053
std       2.263194      265.532844   253.159210     5.395200       1.349442
min       0.000000        0.000000     0.000000     8.880000       0.010000
25%       0.000000        0.000000     0.000000    22.620000       1.670000
50%       0.001058        6.129226     6.335000    26.780000       2.380000
75%       3.258698      363.269483   354.457500    29.430000       3.310000
max       8.508586     1028.860455   953.450000    39.390000       9.210000

Mean simulated daily yield: 39.53 kWh/day (~3.95 kWh/kWp/day -- sanity check: monsoon-belt Bangladesh should land roughly 3.5-4.5)


In [ ]:
"""
Step 4: Build the FIXED, documented missingness model from real Bangladesh
load-shedding patterns, and generate the 1-month / 3-month scarce training
splits plus a clean held-out test split.

DERIVATION (cite this paragraph directly in your paper's Methodology section):
Multiple 2023-2024 news reports citing BPDB/REB/PGCB figures consistently
describe rural Bangladesh load shedding as: (a) strongly seasonal -- worst in
the pre-monsoon hot season (Mar-Jun, commonly 7-10+ hours/day, occasionally
10-12 in heatwave crises), elevated through monsoon (Jun-Sep, ~4-6 hours/day),
moderate in post-monsoon autumn (Oct-Nov, ~3-5 hours/day), and mildest in
winter (Dec-Feb, ~2-3 hours/day); and (b) delivered as several shorter
outages per day (commonly reported as roughly 2-5 events, often 0.5-2 hours
each) concentrated in afternoon/evening peak-demand hours, rather than one
continuous block, with a smaller chance of additional outages overnight.
Sources: Bangladesh Post (BPDB statement on 1500-2000MW average deficit);
Dhaka Tribune (rural areas 6-10 hrs/day, "10 to 12 times daily" outages);
Prothom Alo (rural areas up to 10-12 hrs/day in crisis periods, "maximum
three hours of load shedding after midnight"). A more granular, citable
machine-readable source for a future extension of this work is the
"Multi-year dataset on daily electricity demand, generation, load shedding,
and external conditions in Bangladesh" (Data in Brief / ScienceDirect, 2025),
which scrapes daily BPDB reports Nov 2019-Dec 2024.

CRITICAL METHODOLOGICAL RULE: the seasonal-hours table and event-sampling
parameters below are fixed ONCE, with a fixed random seed, before looking at
ANY model results. Do not adjust them after seeing how the transfer model or
baseline performs -- that is exactly the "tuned to flatter the model"
criticism a reviewer would raise. If you genuinely need to change a
parameter, document why in the paper and re-run every experiment with the
new mask, not just the ones that look better.

USAGE:
    pip install pandas numpy
    python 04_build_missingness_and_splits.py
"""

import numpy as np
import pandas as pd

IN_PATH = "/kaggle/working/bangladesh_target_pv_simulated.csv"
RANDOM_SEED = 42  # fixed once. Do not change after seeing results.

# Average outage HOURS PER DAY by month, grounded in the news reporting cited
# above. These are deliberately round, defensible numbers -- not fit to data.
SEASONAL_MEAN_OUTAGE_HOURS = {
    1: 2.5, 2: 2.5,            # winter: mildest
    3: 7.0, 4: 8.5, 5: 8.5, 6: 7.0,   # pre-monsoon hot season: worst
    7: 5.5, 8: 5.0, 9: 5.0,     # monsoon: elevated but improved from peak
    10: 4.0, 11: 3.0, 12: 2.5,  # post-monsoon / early winter: moderate to mild
}

MIN_EVENTS_PER_DAY = 2
MAX_EVENTS_PER_DAY = 5
MIN_EVENT_HOURS = 1
MAX_EVENT_HOURS = 3

# Hour-of-day weights for when an outage event is more likely to START.
# Peak-shaving load shedding concentrates in afternoon/evening; a smaller
# residual chance exists overnight (per Prothom Alo: "maximum three hours of
# load shedding occurs after midnight").
HOUR_WEIGHTS = np.array([
    1, 1, 1, 1, 1, 1,            # 00-05: low overnight chance
    2, 3, 3, 3, 3, 4,            # 06-11: rising daytime chance
    5, 6, 7, 8, 8, 8,            # 12-17: afternoon peak
    7, 6, 5, 3, 2, 1,            # 18-23: evening tapering off
], dtype=float)
HOUR_WEIGHTS = HOUR_WEIGHTS / HOUR_WEIGHTS.sum()

# Fixed test window: held out, clean (no missingness) for fair evaluation.
TEST_START = "2023-12-01"
TEST_END = "2023-12-31 23:00"
# Scarce training windows: immediately precede the test window.
TRAIN_1MO_START = "2023-11-01"
TRAIN_1MO_END = "2023-11-30 23:00"
TRAIN_3MO_START = "2023-09-01"
TRAIN_3MO_END = "2023-11-30 23:00"


def build_outage_mask(index, rng):
    """Returns a boolean Series, True = hour lost to a simulated outage."""
    mask = pd.Series(False, index=index)
    for day in pd.date_range(index.min().normalize(), index.max().normalize(), freq="1D"):
        month = day.month
        target_hours = SEASONAL_MEAN_OUTAGE_HOURS[month] + rng.normal(0, 1.0)
        target_hours = float(np.clip(target_hours, 0, 14))

        day_hours = pd.date_range(day, day + pd.Timedelta(hours=23), freq="1h")
        day_hours = day_hours.intersection(mask.index)

        attempts = 0
        while attempts < 30:
            # Always measure actual current coverage directly from the mask --
            # never trust a separately-incremented counter, since overlapping
            # events would silently double-count and stop placement too early.
            current_hours = mask.loc[day_hours].sum() if len(day_hours) else 0
            if current_hours >= target_hours:
                break
            attempts += 1
            start_hour = rng.choice(24, p=HOUR_WEIGHTS)
            duration = int(rng.integers(MIN_EVENT_HOURS, MAX_EVENT_HOURS + 1))
            remaining = target_hours - current_hours
            # Don't let the final event of the day overshoot more than
            # necessary -- still at least 1 whole hour (our data is hourly).
            duration = max(1, min(duration, int(np.ceil(remaining))))
            # No modulo here: plain Timedelta arithmetic correctly rolls an
            # event starting near 23:00 into the next calendar day, instead
            # of wrapping it back onto the same day's early-morning hours.
            for h in range(duration):
                ts = day + pd.Timedelta(hours=int(start_hour) + h)
                if ts in mask.index:
                    mask.loc[ts] = True
    return mask


def main():
    df = pd.read_csv(IN_PATH, index_col=0, parse_dates=True)
    # The CSV from step 3 is in UTC (per step 1). Convert to local time so the
    # outage model's hour-of-day weighting matches real Bangladesh clock time.
    if df.index.tz is None:
        df.index = df.index.tz_localize("UTC")
    df.index = df.index.tz_convert("Asia/Dhaka")

    rng = np.random.default_rng(RANDOM_SEED)
    outage_mask = build_outage_mask(df.index, rng)
    outage_mask.to_frame("is_outage").to_csv("bangladesh_outage_mask.csv")

    realized_pct = outage_mask.mean() * 100
    print(f"Realized overall missingness: {realized_pct:.1f}% of hours")
    print("\nRealized vs. target avg outage hours/day by month:")
    for month in range(1, 13):
        days_in_month = outage_mask[outage_mask.index.month == month].index.normalize().nunique()
        if days_in_month == 0:
            continue
        realized_hours_per_day = outage_mask[outage_mask.index.month == month].sum() / days_in_month
        print(f"  month {month:2d}: target {SEASONAL_MEAN_OUTAGE_HOURS[month]:.1f}h/day, "
              f"realized {realized_hours_per_day:.1f}h/day")

    def make_split(start, end, apply_mask):
        sl = df.loc[start:end].copy()
        if apply_mask:
            local_mask = outage_mask.loc[sl.index]
            sl.loc[local_mask, :] = np.nan
            pct = local_mask.mean() * 100
            print(f"\n{start} to {end}: {len(sl)} rows, {pct:.1f}% masked as missing")
        else:
            print(f"\n{start} to {end}: {len(sl)} rows, clean (no missingness applied)")
        return sl

    train_1mo = make_split(TRAIN_1MO_START, TRAIN_1MO_END, apply_mask=True)
    train_3mo = make_split(TRAIN_3MO_START, TRAIN_3MO_END, apply_mask=True)
    test_clean = make_split(TEST_START, TEST_END, apply_mask=False)

    train_1mo.to_csv("bangladesh_train_1month_masked.csv")
    train_3mo.to_csv("bangladesh_train_3month_masked.csv")
    test_clean.to_csv("bangladesh_test_clean.csv")
    print("\nSaved: bangladesh_train_1month_masked.csv, "
          "bangladesh_train_3month_masked.csv, bangladesh_test_clean.csv, "
          "bangladesh_outage_mask.csv")


if __name__ == "__main__":
    main()

Realized overall missingness: 22.8% of hours

Realized vs. target avg outage hours/day by month:
  month  1: target 2.5h/day, realized 2.8h/day
  month  2: target 2.5h/day, realized 2.9h/day
  month  3: target 7.0h/day, realized 7.4h/day
  month  4: target 8.5h/day, realized 9.2h/day
  month  5: target 8.5h/day, realized 9.1h/day
  month  6: target 7.0h/day, realized 7.2h/day
  month  7: target 5.5h/day, realized 5.8h/day
  month  8: target 5.0h/day, realized 5.3h/day
  month  9: target 5.0h/day, realized 5.1h/day
  month 10: target 4.0h/day, realized 4.7h/day
  month 11: target 3.0h/day, realized 3.1h/day
  month 12: target 2.5h/day, realized 2.8h/day

2023-11-01 to 2023-11-30 23:00: 720 rows, 13.1% masked as missing

2023-09-01 to 2023-11-30 23:00: 2184 rows, 18.0% masked as missing

2023-12-01 to 2023-12-31 23:00: 744 rows, clean (no missingness applied)

Saved: bangladesh_train_1month_masked.csv, bangladesh_train_3month_masked.csv, bangladesh_test_clean.csv, bangladesh_outage_mask.

In [ ]:
"""
Day 4-5: Train scratch baseline + pretrain on DKASC + fine-tune (transfer).

DESIGN DECISIONS (document these in your paper):
- Architecture: 2-layer stacked LSTM. Not a novelty contribution -- the
  contribution is the transfer+UQ framework, not the network design.
- Features: GHI + air_temperature + hour_sin + hour_cos. Wind speed dropped
  from DKASC (53% missing). This 4-feature set is clean in both domains.
- Sequence length: 24h lookback to predict the next hour's AC power.
- Normalisation: StandardScaler fit ONLY on training data, never on val/test.
  This is non-negotiable -- fitting on val/test is data leakage.
- Hyperparameter search: same 12-combination grid, same number of epochs,
  same early-stopping patience for BOTH scratch and fine-tune. This is the
  single most important thing a reviewer checks for a fair comparison.
- Fine-tune strategy: full fine-tuning (all layers), lower LR range than
  scratch to avoid catastrophic forgetting of the pretrained representation.
- Missing data: NaN rows in training splits are DROPPED before sequencing.
  Do not interpolate -- that would fabricate data and obscure the missingness
  effect we are specifically studying.

OUTPUTS (saved to same folder as the script):
  pretrained_backbone.pt       -- DKASC-pretrained weights (best val loss)
  scratch_1mo_best.pt          -- best scratch model on 1-month split
  scratch_3mo_best.pt          -- best scratch model on 3-month split
  transfer_1mo_best.pt         -- best fine-tuned model on 1-month split
  transfer_3mo_best.pt         -- best fine-tuned model on 3-month split
  training_results.csv         -- all HP search results for every model

USAGE:
  pip install torch scikit-learn pandas numpy
  python 05_train_models.py
  (runs in ~15-30 min on CPU, faster on GPU -- Kaggle T4 recommended)
"""

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import StandardScaler
from itertools import product
import copy, time, warnings
warnings.filterwarnings("ignore")

# ── PATHS (FIXED FOR KAGGLE) ──────────────────────────────────────────────────
# Raw input data on Kaggle lives in /kaggle/input/. Check your exact dataset folder name!
# Replace 'your-dkasc-dataset-folder' with the exact name of your uploaded Kaggle input folder.
DKASC_PATH    = "/kaggle/working/dkasc_clean_hourly.csv"

# Generated training files from Step 4 sit in your writable working directory
TRAIN_1MO     = "/kaggle/working/bangladesh_train_1month_masked.csv"
TRAIN_3MO     = "/kaggle/working/bangladesh_train_3month_masked.csv"
TEST_CLEAN    = "/kaggle/working/bangladesh_test_clean.csv"

# Outputs will be cleanly written directly to your writable workspace
RESULTS_CSV   = "/kaggle/working/training_results.csv"

# ── ARCHITECTURE & TRAINING CONSTANTS ────────────────────────────────────────
SEQ_LEN        = 24       # 24h lookback
BATCH_SIZE     = 64
MAX_EPOCHS     = 100
PATIENCE       = 10       # early stopping patience (same for all models)
DEVICE         = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Features used in both domains. Must be present (after renaming) in all CSVs.
# DKASC columns after script 2: ghi, temp_air_c, ac_power_kw
# Bangladesh columns after script 3: ghi_wm2, temp_air_c, ac_power_kw
FEATURE_COLS   = ["ghi", "temp_air_c"]   # + hour_sin/hour_cos added below
TARGET_COL     = "ac_power_kw"

# ── HP SEARCH GRID ────────────────────────────────────────────────────────────
# Same grid for scratch and fine-tune (for fair comparison).
# Fine-tune uses a lower LR range to avoid catastrophic forgetting.
SCRATCH_GRID = {
    "hidden_size": [32, 64],
    "num_layers":  [1, 2],
    "dropout":     [0.1, 0.2],
    "lr":          [1e-3, 1e-4],
}
PRETRAIN_GRID = {            # same arch search as scratch (also fair)
    "hidden_size": [32, 64],
    "num_layers":  [1, 2],
    "dropout":     [0.1, 0.2],
    "lr":          [1e-3, 1e-4],
}
FINETUNE_LR   = [5e-4, 1e-4, 1e-5]   # lower LR range for fine-tuning only


# ══════════════════════════════════════════════════════════════════════════════
# MODEL
# ══════════════════════════════════════════════════════════════════════════════

class LSTMForecaster(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers, dropout):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            dropout=dropout if num_layers > 1 else 0.0,
            batch_first=True,
        )
        self.head = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(hidden_size, 1),
        )

    def forward(self, x):
        out, _ = self.lstm(x)
        return self.head(out[:, -1, :]).squeeze(-1)


# ══════════════════════════════════════════════════════════════════════════════
# DATA HELPERS
# ══════════════════════════════════════════════════════════════════════════════

def add_time_features(df):
    """Add cyclical hour-of-day features. Works on any tz-aware or tz-naive index."""
    hour = df.index.hour.to_numpy().astype(float)
    df = df.copy()
    df["hour_sin"] = np.sin(2 * np.pi * hour / 24)
    df["hour_cos"] = np.cos(2 * np.pi * hour / 24)
    return df


def load_dkasc(path):
    df = pd.read_csv(path, index_col=0, parse_dates=True)
    # Rename GHI column to the shared name
    if "ghi" not in df.columns and "ghi_wm2" not in df.columns:
        raise ValueError("DKASC file missing ghi column -- check script 2 output")
    df = df.rename(columns={"ghi": "ghi", "ghi_wm2": "ghi"})
    df = df[[TARGET_COL] + FEATURE_COLS].dropna()
    df = add_time_features(df)
    return df


def load_bangladesh(path):
    df = pd.read_csv(path, index_col=0, parse_dates=True)
    # Bangladesh csv from script 3 uses ghi_wm2; rename to match shared name
    df = df.rename(columns={"ghi_wm2": "ghi"})
    df = df[[TARGET_COL] + FEATURE_COLS].copy()
    df = add_time_features(df)
    # Drop NaN rows (masked outage hours) -- do NOT interpolate
    n_before = len(df)
    df = df.dropna()
    print(f"  Dropped {n_before - len(df)} NaN rows ({100*(n_before-len(df))/n_before:.1f}%); "
          f"{len(df)} rows remain for training")
    return df


def make_sequences(df, scaler, fit_scaler=False):
    """Build (X, y) sequence tensors. Scaler fit only when fit_scaler=True."""
    feat_cols = FEATURE_COLS + ["hour_sin", "hour_cos"]
    X_raw = df[feat_cols].values.astype(np.float32)
    y_raw = df[TARGET_COL].values.astype(np.float32)

    if fit_scaler:
        X_raw = scaler.fit_transform(X_raw)
    else:
        X_raw = scaler.transform(X_raw)

    Xs, ys = [], []
    for i in range(SEQ_LEN, len(X_raw)):
        Xs.append(X_raw[i - SEQ_LEN: i])
        ys.append(y_raw[i])
    if not Xs:
        return None, None
    return (torch.tensor(np.array(Xs), dtype=torch.float32),
            torch.tensor(np.array(ys),  dtype=torch.float32))


def train_val_split(X, y, val_frac=0.2):
    n = len(X)
    split = int(n * (1 - val_frac))
    return X[:split], y[:split], X[split:], y[split:]


def make_loader(X, y, shuffle=True):
    ds = TensorDataset(X.to(DEVICE), y.to(DEVICE))
    return DataLoader(ds, batch_size=BATCH_SIZE, shuffle=shuffle)


# ══════════════════════════════════════════════════════════════════════════════
# TRAINING LOOP
# ══════════════════════════════════════════════════════════════════════════════

def run_one(model, train_loader, val_loader, lr, tag=""):
    optimiser = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.MSELoss()
    best_val  = float("inf")
    best_state = None
    no_improve = 0

    for epoch in range(1, MAX_EPOCHS + 1):
        model.train()
        for xb, yb in train_loader:
            optimiser.zero_grad()
            loss = criterion(model(xb), yb)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimiser.step()

        model.eval()
        val_losses = []
        with torch.no_grad():
            for xb, yb in val_loader:
                val_losses.append(criterion(model(xb), yb).item())
        val_loss = float(np.mean(val_losses))

        if val_loss < best_val - 1e-6:
            best_val   = val_loss
            best_state = copy.deepcopy(model.state_dict())
            no_improve = 0
        else:
            no_improve += 1
            if no_improve >= PATIENCE:
                break

    model.load_state_dict(best_state)
    return model, best_val


def grid_search(X_tr, y_tr, X_val, y_val, grid, input_size, tag, fixed_arch=None, lr_list=None):
    """
    Run HP search. If fixed_arch is provided (dict with hidden_size,
    num_layers, dropout), only search over lr_list -- used for fine-tuning
    where we must keep the same architecture as the pretrained model.
    """
    tr_loader  = make_loader(X_tr, y_tr, shuffle=True)
    val_loader = make_loader(X_val, y_val, shuffle=False)

    if fixed_arch is not None:
        combos = [{"lr": lr, **fixed_arch} for lr in lr_list]
    else:
        keys   = list(grid.keys())
        vals   = list(grid.values())
        combos = [dict(zip(keys, v)) for v in product(*vals)]

    best_val, best_model, best_hp = float("inf"), None, None
    rows = []
    print(f"\n  [{tag}] searching {len(combos)} HP combos on {DEVICE}...")

    for i, hp in enumerate(combos, 1):
        model = LSTMForecaster(
            input_size=input_size,
            hidden_size=hp["hidden_size"],
            num_layers=hp["num_layers"],
            dropout=hp["dropout"],
        ).to(DEVICE)

        # For fine-tuning: load pretrained weights then update lr only
        if "pretrained_state" in hp:
            model.load_state_dict(hp["pretrained_state"])

        model, val_loss = run_one(model, tr_loader, val_loader, hp["lr"],
                                  tag=f"{tag} combo {i}/{len(combos)}")
        rows.append({**hp, "tag": tag, "val_mse": val_loss})
        print(f"    combo {i:2d}/{len(combos)}: "
              f"h={hp['hidden_size']} l={hp['num_layers']} "
              f"d={hp['dropout']} lr={hp['lr']:.0e} → val_mse={val_loss:.4f}")

        if val_loss < best_val:
            best_val   = val_loss
            best_model = copy.deepcopy(model)
            best_hp    = hp

    print(f"  [{tag}] BEST val_mse={best_val:.4f}  hp={best_hp}")
    return best_model, best_hp, rows


# ══════════════════════════════════════════════════════════════════════════════
# MAIN
# ══════════════════════════════════════════════════════════════════════════════

def main():
    t0 = time.time()
    all_rows = []
    input_size = len(FEATURE_COLS) + 2   # +2 for hour_sin, hour_cos

    # ── 1. PRETRAIN ON DKASC ─────────────────────────────────────────────────
    print("\n=== STEP 1: Pretrain backbone on DKASC ===")
    df_dkasc = load_dkasc(DKASC_PATH)
    print(f"  DKASC: {len(df_dkasc)} rows after dropna")

    scaler_dkasc = StandardScaler()
    X_d, y_d = make_sequences(df_dkasc, scaler_dkasc, fit_scaler=True)
    X_dtr, y_dtr, X_dval, y_dval = train_val_split(X_d, y_d, val_frac=0.2)

    pretrained_model, best_pretrain_hp, rows = grid_search(
        X_dtr, y_dtr, X_dval, y_dval,
        grid=PRETRAIN_GRID, input_size=input_size, tag="PRETRAIN_DKASC"
    )
    all_rows.extend(rows)
    torch.save({
        "state_dict": pretrained_model.state_dict(),
        "hp": best_pretrain_hp,
        "scaler_mean": scaler_dkasc.mean_,
        "scaler_scale": scaler_dkasc.scale_,
    }, "pretrained_backbone.pt")
    print("  Saved: pretrained_backbone.pt")

    # ── 2. TRAIN SCRATCH AND FINE-TUNE ON EACH SPLIT ─────────────────────────
    splits = {
        "1mo": TRAIN_1MO,
        "3mo": TRAIN_3MO,
    }

    for split_name, split_path in splits.items():
        print(f"\n=== STEP 2: Split = {split_name} ===")
        df_bd = load_bangladesh(split_path)

        if len(df_bd) < SEQ_LEN + 10:
            print(f"  WARNING: too few rows ({len(df_bd)}) after masking -- skipping")
            continue

        # Fit a SEPARATE scaler for this target split (never use DKASC scaler
        # on Bangladesh data -- different solar climate, different units range).
        scaler_bd = StandardScaler()
        X_bd, y_bd = make_sequences(df_bd, scaler_bd, fit_scaler=True)
        X_tr, y_tr, X_val, y_val = train_val_split(X_bd, y_bd, val_frac=0.2)
        print(f"  Sequences: {len(X_tr)} train, {len(X_val)} val")

        # ── SCRATCH ──────────────────────────────────────────────────────────
        print(f"\n  -- Scratch baseline ({split_name}) --")
        scratch_model, scratch_hp, rows = grid_search(
            X_tr, y_tr, X_val, y_val,
            grid=SCRATCH_GRID, input_size=input_size, tag=f"SCRATCH_{split_name}"
        )
        all_rows.extend(rows)
        torch.save({
            "state_dict":   scratch_model.state_dict(),
            "hp":           scratch_hp,
            "scaler_mean":  scaler_bd.mean_,
            "scaler_scale": scaler_bd.scale_,
        }, f"scratch_{split_name}_best.pt")
        print(f"  Saved: scratch_{split_name}_best.pt")

        # ── FINE-TUNE (transfer) ─────────────────────────────────────────────
        print(f"\n  -- Fine-tune transfer ({split_name}) --")
        ckpt = torch.load("pretrained_backbone.pt", map_location=DEVICE, weights_only=False)
        pretrain_arch = {
            "hidden_size": ckpt["hp"]["hidden_size"],
            "num_layers":  ckpt["hp"]["num_layers"],
            "dropout":     ckpt["hp"]["dropout"],
        }

        # Build combos manually: same architecture, vary only LR
        combos_ft = [
            {"lr": lr, **pretrain_arch, "pretrained_state": ckpt["state_dict"]}
            for lr in FINETUNE_LR
        ]
        tr_loader  = make_loader(X_tr, y_tr, shuffle=True)
        val_loader = make_loader(X_val, y_val, shuffle=False)

        best_val_ft, best_ft_model, best_ft_hp = float("inf"), None, None
        ft_rows = []
        print(f"  [{split_name} TRANSFER] searching {len(combos_ft)} LR combos...")
        for i, hp in enumerate(combos_ft, 1):
            model_ft = LSTMForecaster(
                input_size=input_size,
                hidden_size=hp["hidden_size"],
                num_layers=hp["num_layers"],
                dropout=hp["dropout"],
            ).to(DEVICE)
            model_ft.load_state_dict(hp["pretrained_state"])
            model_ft, val_loss = run_one(model_ft, tr_loader, val_loader, hp["lr"])
            ft_rows.append({"tag": f"TRANSFER_{split_name}", **{k:v for k,v in hp.items()
                                                                  if k != "pretrained_state"},
                            "val_mse": val_loss})
            print(f"    LR={hp['lr']:.0e} → val_mse={val_loss:.4f}")
            if val_loss < best_val_ft:
                best_val_ft = val_loss
                best_ft_model = copy.deepcopy(model_ft)
                best_ft_hp = {k:v for k,v in hp.items() if k != "pretrained_state"}

        all_rows.extend(ft_rows)
        print(f"  [TRANSFER_{split_name}] BEST val_mse={best_val_ft:.4f} hp={best_ft_hp}")
        torch.save({
            "state_dict":   best_ft_model.state_dict(),
            "hp":           best_ft_hp,
            "scaler_mean":  scaler_bd.mean_,
            "scaler_scale": scaler_bd.scale_,
        }, f"transfer_{split_name}_best.pt")
        print(f"  Saved: transfer_{split_name}_best.pt")

    # ── SAVE HP SEARCH RESULTS ────────────────────────────────────────────────
    pd.DataFrame(all_rows).to_csv(RESULTS_CSV, index=False)
    print(f"\n=== DONE in {(time.time()-t0)/60:.1f} min ===")
    print(f"Saved {len(all_rows)} HP search results to {RESULTS_CSV}")
    print("\nFiles produced:")
    for f in ["pretrained_backbone.pt",
              "scratch_1mo_best.pt", "scratch_3mo_best.pt",
              "transfer_1mo_best.pt", "transfer_3mo_best.pt",
              RESULTS_CSV]:
        print(f"  {f}")
    print("\nNext step: run 06_cqr_evaluation.py to attach CQR and evaluate all 4 variants.")


if __name__ == "__main__":
    main()


=== STEP 1: Pretrain backbone on DKASC ===
  DKASC: 146092 rows after dropna

  [PRETRAIN_DKASC] searching 16 HP combos on cuda...
    combo  1/16: h=32 l=1 d=0.1 lr=1e-03 → val_mse=0.1306
    combo  2/16: h=32 l=1 d=0.1 lr=1e-04 → val_mse=0.1438
    combo  3/16: h=32 l=1 d=0.2 lr=1e-03 → val_mse=0.1256
    combo  4/16: h=32 l=1 d=0.2 lr=1e-04 → val_mse=0.1385
    combo  5/16: h=32 l=2 d=0.1 lr=1e-03 → val_mse=0.1238
    combo  6/16: h=32 l=2 d=0.1 lr=1e-04 → val_mse=0.1418
    combo  7/16: h=32 l=2 d=0.2 lr=1e-03 → val_mse=0.1226
    combo  8/16: h=32 l=2 d=0.2 lr=1e-04 → val_mse=0.1334
    combo  9/16: h=64 l=1 d=0.1 lr=1e-03 → val_mse=0.1343
    combo 10/16: h=64 l=1 d=0.1 lr=1e-04 → val_mse=0.1398
    combo 11/16: h=64 l=1 d=0.2 lr=1e-03 → val_mse=0.1316
    combo 12/16: h=64 l=1 d=0.2 lr=1e-04 → val_mse=0.1458
    combo 13/16: h=64 l=2 d=0.1 lr=1e-03 → val_mse=0.1253
    combo 14/16: h=64 l=2 d=0.1 lr=1e-04 → val_mse=0.1354
    combo 15/16: h=64 l=2 d=0.2 lr=1e-03 → val_mse=0.123

In [ ]:
"""
Day 6: Attach CQR (Conformalized Quantile Regression) to the scratch and
transfer models, then produce the final 4-variant evaluation on the clean
held-out December test set.

WHAT THIS BUILDS:
  - 4 quantile models (lower=5%, upper=95% -> 90% target coverage), one per
    {scratch, transfer} x {1mo, 3mo}, trained with pinball loss.
  - scratch+CQR quantile models: random init, trained on that split's data
    (mirrors how the scratch point model was built).
  - transfer+CQR quantile models: LSTM backbone initialized from
    pretrained_backbone.pt (DKASC), head randomly initialized (different
    output shape), then fine-tuned on that split's data (mirrors how the
    transfer point model was built).
  - NO new architecture search: each quantile model reuses the exact
    hidden_size/num_layers/dropout/lr already locked in by the corresponding
    point model's checkpoint from script 05. This avoids introducing new,
    reviewer-visible tuning decisions at this stage.
  - Calibration: split-conformal using the SAME validation sequences used
    for early-stopping in script 05 (not the training data, not the test
    set). Using the validation split as the calibration set is a standard,
    documented simplification in small-sample conformal prediction -- state
    this explicitly in your Methodology.

THE 4 CORE VARIANTS reported in the final table:
  scratch        -> point model only (RMSE, MAE, skill score)
  scratch+CQR    -> scratch-recipe quantile model, conformalized (coverage, width, pinball)
  transfer       -> point model only (RMSE, MAE, skill score)
  transfer+CQR   -> transfer-recipe quantile model, conformalized (coverage, width, pinball)
  ... each x {1mo, 3mo} = 8 rows total in final_evaluation_results.csv

NOT INCLUDED HERE (deliberately deferred, per the original risk-mitigation
plan): the downstream dispatch-decision-error metric. Add it only after this
core comparison is validated and if time remains -- it is a stretch goal,
not a commitment.

USAGE:
  pip install torch scikit-learn pandas numpy
  python 06_cqr_evaluation.py
"""

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
import copy, time, warnings
warnings.filterwarnings("ignore")

import os

# ── PATHS (FIXED FOR KAGGLE) ──────────────────────────────────────────────────
BASE_DIR = "/kaggle/working"

TEST_CLEAN    = os.path.join(BASE_DIR, "bangladesh_test_clean.csv")

TRAIN_PATHS   = {
    "1mo": os.path.join(BASE_DIR, "bangladesh_train_1month_masked.csv"),
    "3mo": os.path.join(BASE_DIR, "bangladesh_train_3month_masked.csv")
}

PRETRAIN_CKPT = os.path.join(BASE_DIR, "pretrained_backbone.pt")

POINT_CKPTS   = {
    ("scratch", "1mo"):  os.path.join(BASE_DIR, "scratch_1mo_best.pt"),
    ("scratch", "3mo"):  os.path.join(BASE_DIR, "scratch_3mo_best.pt"),
    ("transfer", "1mo"): os.path.join(BASE_DIR, "transfer_1mo_best.pt"),
    ("transfer", "3mo"): os.path.join(BASE_DIR, "transfer_3mo_best.pt")
}

OUT_CSV       = os.path.join(BASE_DIR, "final_evaluation_results.csv")

SEQ_LEN      = 24
BATCH_SIZE   = 64
MAX_EPOCHS   = 100
PATIENCE     = 10
DEVICE       = torch.device("cuda" if torch.cuda.is_available() else "cpu")
FEATURE_COLS = ["ghi", "temp_air_c"]
TARGET_COL   = "ac_power_kw"
QUANTILES    = [0.05, 0.95]   # 90% target coverage
ALPHA        = 1 - (QUANTILES[1] - QUANTILES[0])  # 0.10


# ══════════════════════════════════════════════════════════════════════════════
# DATA HELPERS (mirrors script 05 -- duplicated for standalone use)
# ══════════════════════════════════════════════════════════════════════════════

def add_time_features(df):
    hour = df.index.hour.to_numpy().astype(float)
    df = df.copy()
    df["hour_sin"] = np.sin(2 * np.pi * hour / 24)
    df["hour_cos"] = np.cos(2 * np.pi * hour / 24)
    return df


def load_bangladesh(path, drop_na=True):
    df = pd.read_csv(path, index_col=0, parse_dates=True)
    df = df.rename(columns={"ghi_wm2": "ghi"})
    df = df[[TARGET_COL] + FEATURE_COLS].copy()
    df = add_time_features(df)
    if drop_na:
        n_before = len(df)
        df = df.dropna()
        print(f"  Dropped {n_before - len(df)} NaN rows "
              f"({100*(n_before-len(df))/max(n_before,1):.1f}%); {len(df)} rows remain")
    return df


def scale_with(df, mean_, scale_):
    feat_cols = FEATURE_COLS + ["hour_sin", "hour_cos"]
    X_raw = df[feat_cols].values.astype(np.float32)
    X_scaled = (X_raw - mean_) / scale_
    y_raw = df[TARGET_COL].values.astype(np.float32)
    return X_scaled, y_raw


def make_sequences_from_arrays(X_scaled, y_raw):
    Xs, ys = [], []
    for i in range(SEQ_LEN, len(X_scaled)):
        Xs.append(X_scaled[i - SEQ_LEN: i])
        ys.append(y_raw[i])
    if not Xs:
        return None, None
    return (torch.tensor(np.array(Xs), dtype=torch.float32),
            torch.tensor(np.array(ys),  dtype=torch.float32))


def train_val_split(X, y, val_frac=0.2):
    n = len(X)
    split = int(n * (1 - val_frac))
    return X[:split], y[:split], X[split:], y[split:]


def make_loader(X, y, shuffle=True):
    ds = TensorDataset(X.to(DEVICE), y.to(DEVICE))
    return DataLoader(ds, batch_size=BATCH_SIZE, shuffle=shuffle)


# ══════════════════════════════════════════════════════════════════════════════
# MODELS
# ══════════════════════════════════════════════════════════════════════════════

class QuantileLSTM(nn.Module):
    """Same backbone shape as the point LSTMForecaster, 2-unit output head
    (lower quantile, upper quantile) instead of 1."""
    def __init__(self, input_size, hidden_size, num_layers, dropout, n_quantiles=2):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size=input_size, hidden_size=hidden_size, num_layers=num_layers,
            dropout=dropout if num_layers > 1 else 0.0, batch_first=True,
        )
        self.head = nn.Sequential(nn.Dropout(dropout), nn.Linear(hidden_size, n_quantiles))

    def forward(self, x):
        out, _ = self.lstm(x)
        return self.head(out[:, -1, :])


def pinball_loss(preds, y, quantiles):
    """preds: (batch, n_quantiles); y: (batch,)"""
    losses = []
    for i, q in enumerate(quantiles):
        errors = y - preds[:, i]
        losses.append(torch.maximum((q - 1) * errors, q * errors))
    return torch.stack(losses, dim=1).mean()


def load_partial_lstm_weights(model, pretrained_state):
    """Copy only matching-shape LSTM weights from a point model's checkpoint
    into a quantile model. The head is a different shape (1 vs 2 outputs) so
    it is left randomly initialized -- it must learn the quantile mapping
    from scratch regardless of the point model's learned head."""
    own_state = model.state_dict()
    copied = 0
    for k, v in pretrained_state.items():
        if k.startswith("lstm") and k in own_state and own_state[k].shape == v.shape:
            own_state[k] = v
            copied += 1
    model.load_state_dict(own_state)
    return copied


def train_quantile_model(model, X_tr, y_tr, X_val, y_val, lr):
    optimiser = torch.optim.Adam(model.parameters(), lr=lr)
    tr_loader  = make_loader(X_tr, y_tr, shuffle=True)
    val_loader = make_loader(X_val, y_val, shuffle=False)
    best_val, best_state, no_improve = float("inf"), None, 0

    for epoch in range(1, MAX_EPOCHS + 1):
        model.train()
        for xb, yb in tr_loader:
            optimiser.zero_grad()
            loss = pinball_loss(model(xb), yb, QUANTILES)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimiser.step()

        model.eval()
        val_losses = []
        with torch.no_grad():
            for xb, yb in val_loader:
                val_losses.append(pinball_loss(model(xb), yb, QUANTILES).item())
        val_loss = float(np.mean(val_losses))

        if val_loss < best_val - 1e-6:
            best_val, best_state, no_improve = val_loss, copy.deepcopy(model.state_dict()), 0
        else:
            no_improve += 1
            if no_improve >= PATIENCE:
                break

    model.load_state_dict(best_state)
    return model, best_val


# ══════════════════════════════════════════════════════════════════════════════
# CONFORMAL CALIBRATION + METRICS
# ══════════════════════════════════════════════════════════════════════════════

def conformal_calibrate(model, X_cal, y_cal, alpha):
    model.eval()
    with torch.no_grad():
        preds = model(X_cal.to(DEVICE)).cpu().numpy()
    y = y_cal.numpy()
    q_low, q_high = preds[:, 0], preds[:, 1]
    scores = np.maximum(q_low - y, y - q_high)
    n = len(scores)
    level = min(1.0, np.ceil((n + 1) * (1 - alpha)) / n)
    q_hat = np.quantile(scores, level, method="higher")
    return float(q_hat)


def evaluate_quantile_model(model, X_test, y_test, q_hat):
    model.eval()
    with torch.no_grad():
        preds = model(X_test.to(DEVICE)).cpu().numpy()
    y = y_test.numpy()
    lower = preds[:, 0] - q_hat
    upper = preds[:, 1] + q_hat
    upper = np.maximum(upper, lower)  # safety: enforce non-crossing after calibration

    coverage = float(np.mean((y >= lower) & (y <= upper)))
    avg_width = float(np.mean(upper - lower))
    pb = (np.mean(np.maximum(QUANTILES[0] * (y - lower), (QUANTILES[0] - 1) * (y - lower))) +
          np.mean(np.maximum(QUANTILES[1] * (y - upper), (QUANTILES[1] - 1) * (y - upper)))) / 2
    return coverage, avg_width, float(pb)


def evaluate_point_model(model, X_test, y_test):
    model.eval()
    with torch.no_grad():
        preds = model(X_test.to(DEVICE)).cpu().numpy()
    y = y_test.numpy()
    rmse = float(np.sqrt(np.mean((preds - y) ** 2)))
    mae  = float(np.mean(np.abs(preds - y)))

    # Smart persistence: forecast(t) = actual(t-24h), valid after the first day.
    if len(y) > 24:
        persist_pred = y[:-24]
        persist_true = y[24:]
        rmse_persist = float(np.sqrt(np.mean((persist_pred - persist_true) ** 2)))
        skill = 1 - (rmse / rmse_persist) if rmse_persist > 0 else np.nan
    else:
        skill = np.nan
    return rmse, mae, skill


# ══════════════════════════════════════════════════════════════════════════════
# POINT MODEL CLASS (must match script 05 exactly to load checkpoints)
# ══════════════════════════════════════════════════════════════════════════════

class LSTMForecaster(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers, dropout):
        super().__init__()
        self.lstm = nn.LSTM(input_size=input_size, hidden_size=hidden_size,
                             num_layers=num_layers,
                             dropout=dropout if num_layers > 1 else 0.0, batch_first=True)
        self.head = nn.Sequential(nn.Dropout(dropout), nn.Linear(hidden_size, 1))

    def forward(self, x):
        out, _ = self.lstm(x)
        return self.head(out[:, -1, :]).squeeze(-1)


# ══════════════════════════════════════════════════════════════════════════════
# MAIN
# ══════════════════════════════════════════════════════════════════════════════

def main():
    t0 = time.time()
    input_size = len(FEATURE_COLS) + 2
    pretrain_ckpt = torch.load(PRETRAIN_CKPT, map_location=DEVICE, weights_only=False)

    df_test = load_bangladesh(TEST_CLEAN, drop_na=False)
    assert df_test.isna().sum().sum() == 0, "Test set should be clean (no NaN) -- check TEST_CLEAN file"

    results = []

    for split in ["1mo", "3mo"]:
        print(f"\n=== Split = {split} ===")
        df_train = load_bangladesh(TRAIN_PATHS[split], drop_na=True)

        for variant in ["scratch", "transfer"]:
            print(f"\n  -- {variant} ({split}) --")
            point_ckpt = torch.load(POINT_CKPTS[(variant, split)], map_location=DEVICE, weights_only=False)
            mean_, scale_ = point_ckpt["scaler_mean"], point_ckpt["scaler_scale"]
            hp = point_ckpt["hp"]

            # ---- Point model evaluation on test set ----
            point_model = LSTMForecaster(input_size, hp["hidden_size"], hp["num_layers"], hp["dropout"]).to(DEVICE)
            point_model.load_state_dict(point_ckpt["state_dict"])
            X_test_s, y_test_s = scale_with(df_test, mean_, scale_)
            X_test, y_test = make_sequences_from_arrays(X_test_s, y_test_s)
            rmse, mae, skill = evaluate_point_model(point_model, X_test, y_test)
            print(f"    POINT  -> RMSE={rmse:.4f}  MAE={mae:.4f}  skill={skill:.4f}")
            results.append({"variant": variant, "split": split, "metric_type": "point",
                             "rmse": rmse, "mae": mae, "skill_score": skill,
                             "n_test_seq": len(y_test)})

            # ---- Build train/val sequences for THIS split using its own scaler ----
            X_tv_s, y_tv_s = scale_with(df_train, mean_, scale_)
            X_tv, y_tv = make_sequences_from_arrays(X_tv_s, y_tv_s)
            X_tr, y_tr, X_val, y_val = train_val_split(X_tv, y_tv, val_frac=0.2)

            # ---- Quantile model: same locked-in architecture as point model ----
            q_model = QuantileLSTM(input_size, hp["hidden_size"], hp["num_layers"], hp["dropout"]).to(DEVICE)
            if variant == "transfer":
                n_copied = load_partial_lstm_weights(q_model, pretrain_ckpt["state_dict"])
                print(f"    Initialized quantile model from DKASC backbone "
                      f"({n_copied} matching LSTM tensors copied)")
            q_model, val_pb = train_quantile_model(q_model, X_tr, y_tr, X_val, y_val, lr=hp["lr"])
            print(f"    Quantile model trained, val pinball={val_pb:.4f}")

            # ---- Conformal calibration on the validation split ----
            q_hat = conformal_calibrate(q_model, X_val, y_val, ALPHA)
            print(f"    Conformal q_hat={q_hat:.4f}")

            coverage, avg_width, test_pb = evaluate_quantile_model(q_model, X_test, y_test, q_hat)
            target_cov = 1 - ALPHA
            print(f"    {variant}+CQR -> coverage={coverage:.3f} (target {target_cov:.2f})  "
                  f"avg_width={avg_width:.4f}  pinball={test_pb:.4f}")
            results.append({"variant": f"{variant}+CQR", "split": split, "metric_type": "interval",
                             "coverage": coverage, "target_coverage": target_cov,
                             "avg_interval_width": avg_width, "pinball_loss": test_pb,
                             "q_hat": q_hat, "n_test_seq": len(y_test)})

    df_results = pd.DataFrame(results)
    df_results.to_csv(OUT_CSV, index=False)
    print(f"\n=== DONE in {(time.time()-t0)/60:.1f} min ===")
    print(f"Saved final results to {OUT_CSV}")
    print("\n--- Point-forecast summary (RMSE / MAE / skill score) ---")
    print(df_results[df_results.metric_type == "point"]
          [["variant", "split", "rmse", "mae", "skill_score"]].to_string(index=False))
    print("\n--- CQR interval summary (coverage / width / pinball) ---")
    print(df_results[df_results.metric_type == "interval"]
          [["variant", "split", "coverage", "target_coverage", "avg_interval_width", "pinball_loss"]]
          .to_string(index=False))


if __name__ == "__main__":
    main()


=== Split = 1mo ===
  Dropped 94 NaN rows (13.1%); 626 rows remain

  -- scratch (1mo) --
    POINT  -> RMSE=0.6593  MAE=0.3754  skill=-0.0329
    Quantile model trained, val pinball=0.1936
    Conformal q_hat=-0.0034
    scratch+CQR -> coverage=0.985 (target 0.90)  avg_width=6.8376  pinball=0.1735

  -- transfer (1mo) --
    POINT  -> RMSE=0.5030  MAE=0.2740  skill=0.2119
    Initialized quantile model from DKASC backbone (8 matching LSTM tensors copied)
    Quantile model trained, val pinball=0.4162
    Conformal q_hat=3.5071
    transfer+CQR -> coverage=0.988 (target 0.90)  avg_width=8.4586  pinball=0.2134

=== Split = 3mo ===
  Dropped 393 NaN rows (18.0%); 1791 rows remain

  -- scratch (3mo) --
    POINT  -> RMSE=0.4884  MAE=0.2850  skill=0.2349
    Quantile model trained, val pinball=0.0706
    Conformal q_hat=-0.0241
    scratch+CQR -> coverage=0.842 (target 0.90)  avg_width=2.3095  pinball=0.0635

  -- transfer (3mo) --
    POINT  -> RMSE=0.4206  MAE=0.2166  skill=0.3411
    

In [ ]:
"""
Diagnostic: does CQR undercoverage concentrate in the hours most often
removed from the training/calibration set by simulated outages?

WHY THIS MATTERS: if undercoverage is concentrated in exactly the hours the
outage mask disproportionately removes (afternoon/evening, per the
HOUR_WEIGHTS in script 04), that is direct empirical evidence that
structured, outage-driven missingness breaks the conformal exchangeability
assumption -- not just a plausible story, an observed mechanism. This is the
paper's central gap claim, made visible in one table.

Note: the quantile models trained in script 06 were not saved to disk, so
this script retrains them (same locked-in architecture/lr as before, same
procedure) with a fixed seed for reproducibility. Coverage numbers here may
differ slightly from script 06's run (different random init) but should be
qualitatively consistent -- if they aren't, that itself is worth noting.

USAGE:
  python 07_hourly_coverage_diagnostic.py
"""

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
import copy, warnings
warnings.filterwarnings("ignore")

torch.manual_seed(42)
np.random.seed(42)

OUTAGE_MASK_PATH = "bangladesh_outage_mask.csv"
TRAIN_3MO_PATH    = "bangladesh_train_3month_masked.csv"
TEST_CLEAN_PATH   = "bangladesh_test_clean.csv"
CKPT_PATHS = {"scratch": "scratch_3mo_best.pt", "transfer": "transfer_3mo_best.pt"}
PRETRAIN_CKPT = "pretrained_backbone.pt"
OUT_CSV = "hourly_coverage_diagnostic.csv"

SEQ_LEN, BATCH_SIZE, MAX_EPOCHS, PATIENCE = 24, 64, 100, 10
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
FEATURE_COLS, TARGET_COL = ["ghi", "temp_air_c"], "ac_power_kw"
QUANTILES, ALPHA = [0.05, 0.95], 0.10
TRAIN_3MO_START, TRAIN_3MO_END = "2023-09-01", "2023-11-30 23:00"


def add_time_features(df):
    hour = df.index.hour.to_numpy().astype(float)
    df = df.copy()
    df["hour_sin"] = np.sin(2 * np.pi * hour / 24)
    df["hour_cos"] = np.cos(2 * np.pi * hour / 24)
    return df


def load_bangladesh(path, drop_na=True):
    df = pd.read_csv(path, index_col=0, parse_dates=True)
    df = df.rename(columns={"ghi_wm2": "ghi"})
    df = df[[TARGET_COL] + FEATURE_COLS].copy()
    df = add_time_features(df)
    if drop_na:
        df = df.dropna()
    return df


def make_sequences_with_time(df, mean_, scale_):
    feat_cols = FEATURE_COLS + ["hour_sin", "hour_cos"]
    X_raw = df[feat_cols].values.astype(np.float32)
    X_scaled = (X_raw - mean_) / scale_
    y_raw = df[TARGET_COL].values.astype(np.float32)
    Xs, ys, ts = [], [], []
    for i in range(SEQ_LEN, len(X_scaled)):
        Xs.append(X_scaled[i - SEQ_LEN:i]); ys.append(y_raw[i]); ts.append(df.index[i])
    return (torch.tensor(np.array(Xs), dtype=torch.float32),
            torch.tensor(np.array(ys), dtype=torch.float32),
            pd.DatetimeIndex(ts))


def train_val_split(X, y, ts, val_frac=0.2):
    n = len(X); split = int(n * (1 - val_frac))
    return X[:split], y[:split], ts[:split], X[split:], y[split:], ts[split:]


def make_loader(X, y, shuffle=True):
    return DataLoader(TensorDataset(X.to(DEVICE), y.to(DEVICE)), batch_size=BATCH_SIZE, shuffle=shuffle)


class QuantileLSTM(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers, dropout, n_q=2):
        super().__init__()
        self.lstm = nn.LSTM(input_size=input_size, hidden_size=hidden_size, num_layers=num_layers,
                             dropout=dropout if num_layers > 1 else 0.0, batch_first=True)
        self.head = nn.Sequential(nn.Dropout(dropout), nn.Linear(hidden_size, n_q))

    def forward(self, x):
        out, _ = self.lstm(x)
        return self.head(out[:, -1, :])


def pinball_loss(preds, y, quantiles):
    losses = [torch.maximum((q - 1) * (y - preds[:, i]), q * (y - preds[:, i]))
              for i, q in enumerate(quantiles)]
    return torch.stack(losses, dim=1).mean()


def load_partial_lstm_weights(model, pretrained_state):
    own = model.state_dict()
    for k, v in pretrained_state.items():
        if k.startswith("lstm") and k in own and own[k].shape == v.shape:
            own[k] = v
    model.load_state_dict(own)


def train_quantile_model(model, X_tr, y_tr, X_val, y_val, lr):
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    tr_loader, val_loader = make_loader(X_tr, y_tr), make_loader(X_val, y_val, shuffle=False)
    best_val, best_state, no_improve = float("inf"), None, 0
    for _ in range(MAX_EPOCHS):
        model.train()
        for xb, yb in tr_loader:
            opt.zero_grad()
            loss = pinball_loss(model(xb), yb, QUANTILES)
            loss.backward(); nn.utils.clip_grad_norm_(model.parameters(), 1.0); opt.step()
        model.eval()
        with torch.no_grad():
            vl = float(np.mean([pinball_loss(model(xb), yb, QUANTILES).item() for xb, yb in val_loader]))
        if vl < best_val - 1e-6:
            best_val, best_state, no_improve = vl, copy.deepcopy(model.state_dict()), 0
        else:
            no_improve += 1
            if no_improve >= PATIENCE: break
    model.load_state_dict(best_state)
    return model


def conformal_calibrate(model, X_cal, y_cal, alpha):
    model.eval()
    with torch.no_grad():
        preds = model(X_cal.to(DEVICE)).cpu().numpy()
    y = y_cal.numpy()
    scores = np.maximum(preds[:, 0] - y, y - preds[:, 1])
    n = len(scores)
    level = min(1.0, np.ceil((n + 1) * (1 - alpha)) / n)
    return float(np.quantile(scores, level, method="higher"))


def main():
    pretrain_ckpt = torch.load(PRETRAIN_CKPT, map_location=DEVICE, weights_only=False)

    # Outage frequency by hour-of-day, computed from the actual 3-month
    # training window's outage mask (not the whole year) -- this is the
    # exact missingness the calibration set was drawn under.
    mask_df = pd.read_csv(OUTAGE_MASK_PATH, index_col=0, parse_dates=True)
    mask_df.index = mask_df.index.tz_localize("UTC").tz_convert("Asia/Dhaka") if mask_df.index.tz is None else mask_df.index.tz_convert("Asia/Dhaka")
    mask_window = mask_df.loc[TRAIN_3MO_START:TRAIN_3MO_END]
    outage_freq_by_hour = mask_window.groupby(mask_window.index.hour)["is_outage"].mean()

    df_test = load_bangladesh(TEST_CLEAN_PATH, drop_na=False)
    df_train = load_bangladesh(TRAIN_3MO_PATH, drop_na=True)

    coverage_by_variant = {}
    input_size = len(FEATURE_COLS) + 2

    for variant in ["scratch", "transfer"]:
        ckpt = torch.load(CKPT_PATHS[variant], map_location=DEVICE, weights_only=False)
        mean_, scale_, hp = ckpt["scaler_mean"], ckpt["scaler_scale"], ckpt["hp"]

        X_tv, y_tv, ts_tv = make_sequences_with_time(df_train, mean_, scale_)
        X_tr, y_tr, _, X_val, y_val, _ = train_val_split(X_tv, y_tv, ts_tv)

        model = QuantileLSTM(input_size, hp["hidden_size"], hp["num_layers"], hp["dropout"]).to(DEVICE)
        if variant == "transfer":
            load_partial_lstm_weights(model, pretrain_ckpt["state_dict"])
        model = train_quantile_model(model, X_tr, y_tr, X_val, y_val, lr=hp["lr"])
        q_hat = conformal_calibrate(model, X_val, y_val, ALPHA)

        X_test, y_test, ts_test = make_sequences_with_time(df_test, mean_, scale_)
        model.eval()
        with torch.no_grad():
            preds = model(X_test.to(DEVICE)).cpu().numpy()
        lower = preds[:, 0] - q_hat
        upper = np.maximum(preds[:, 1] + q_hat, lower)
        covered = (y_test.numpy() >= lower) & (y_test.numpy() <= upper)

        cov_df = pd.DataFrame({"hour": ts_test.hour, "covered": covered})
        coverage_by_hour = cov_df.groupby("hour")["covered"].mean()
        coverage_by_variant[variant] = coverage_by_hour
        print(f"{variant}+CQR overall test coverage: {covered.mean():.3f} (q_hat={q_hat:.4f})")

    # ── Combine into one diagnostic table ──────────────────────────────────
    table = pd.DataFrame({
        "train_outage_freq": outage_freq_by_hour,
        "scratch_coverage":  coverage_by_variant["scratch"],
        "transfer_coverage": coverage_by_variant["transfer"],
    }).sort_values("train_outage_freq", ascending=False)
    table["scratch_deficit"] = 0.90 - table["scratch_coverage"]
    table["transfer_deficit"] = 0.90 - table["transfer_coverage"]

    print("\n=== Hourly diagnostic (sorted by training-window outage frequency) ===")
    print(table.round(3).to_string())

    corr_scratch = np.corrcoef(table["train_outage_freq"], table["scratch_deficit"])[0, 1]
    corr_transfer = np.corrcoef(table["train_outage_freq"], table["transfer_deficit"])[0, 1]
    print(f"\nCorrelation(outage freq, coverage deficit) -- scratch:  {corr_scratch:.3f}")
    print(f"Correlation(outage freq, coverage deficit) -- transfer: {corr_transfer:.3f}")
    print("\n(A positive correlation means hours more often masked during training "
          "are exactly the hours where test-set coverage falls short of the 90% "
          "target -- i.e. structured missingness breaking conformal exchangeability.)")

    table.to_csv(OUT_CSV)
    print(f"\nSaved {OUT_CSV}")


if __name__ == "__main__":
    main()

scratch+CQR overall test coverage: 0.975 (q_hat=-0.0012)
transfer+CQR overall test coverage: 0.922 (q_hat=-0.0399)

=== Hourly diagnostic (sorted by training-window outage frequency) ===
    train_outage_freq  scratch_coverage  transfer_coverage  scratch_deficit  transfer_deficit
15              0.341             1.000              1.000           -0.100            -0.100
16              0.330             1.000              1.000           -0.100            -0.100
19              0.297             1.000              1.000           -0.100            -0.100
14              0.275             1.000              1.000           -0.100            -0.100
18              0.275             1.000              0.967           -0.100            -0.067
17              0.264             1.000              1.000           -0.100            -0.100
13              0.264             1.000              1.000           -0.100            -0.100
8               0.231             0.867              0.700   

In [ ]:
"""
Day 8: Generate the core results figure for the paper, using the actual
final_evaluation_results.csv numbers (script 06's original run -- the
canonical numbers, not the hourly-diagnostic retrain).

Produces: figure1_point_forecast_results.png and .pdf
  (a) RMSE: scratch vs transfer, 1mo vs 3mo
  (b) Forecast skill score: scratch vs transfer, 1mo vs 3mo, with a zero
      line to highlight the negative-to-positive flip at 1-month scarcity

PDF is vector (preferred for LaTeX \\includegraphics{}), PNG is for Word.
"""

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np

plt.rcParams.update({
    "font.family": "serif",
    "font.size": 9,
    "axes.linewidth": 0.8,
    "axes.edgecolor": "#333333",
})

# ── ACTUAL RESULTS from final_evaluation_results.csv (script 06 original run) ──
splits = ["1-month", "3-month"]
rmse_scratch  = [0.659337, 0.488409]
rmse_transfer = [0.503037, 0.420612]
skill_scratch  = [-0.032913, 0.234861]
skill_transfer = [0.211946, 0.341072]

COLOR_SCRATCH  = "#7a7a7a"   # mid-gray -- distinguishable in B&W print
COLOR_TRANSFER = "#1f5fa8"   # blue
HATCH_SCRATCH  = ""
HATCH_TRANSFER = "//"

x = np.arange(len(splits))
width = 0.32

fig, axes = plt.subplots(1, 2, figsize=(7.0, 3.0))

# Panel (a): RMSE
ax = axes[0]
b1 = ax.bar(x - width/2, rmse_scratch, width, label="Scratch",
            color=COLOR_SCRATCH, edgecolor="black", linewidth=0.6)
b2 = ax.bar(x + width/2, rmse_transfer, width, label="Transfer",
            color=COLOR_TRANSFER, edgecolor="black", linewidth=0.6, hatch=HATCH_TRANSFER)
for bars in (b1, b2):
    for bar in bars:
        h = bar.get_height()
        ax.annotate(f"{h:.3f}", (bar.get_x() + bar.get_width()/2, h),
                    xytext=(0, 2), textcoords="offset points", ha="center", fontsize=7.5)
ax.set_xticks(x); ax.set_xticklabels(splits)
ax.set_ylabel("RMSE (kW)")
ax.set_title("(a) Point-forecast RMSE", fontsize=9)
ax.set_ylim(0, max(rmse_scratch) * 1.25)
ax.spines[["top", "right"]].set_visible(False)

# Panel (b): Skill score
ax = axes[1]
b1 = ax.bar(x - width/2, skill_scratch, width, label="Scratch",
            color=COLOR_SCRATCH, edgecolor="black", linewidth=0.6)
b2 = ax.bar(x + width/2, skill_transfer, width, label="Transfer",
            color=COLOR_TRANSFER, edgecolor="black", linewidth=0.6, hatch=HATCH_TRANSFER)
for bars in (b1, b2):
    for bar in bars:
        h = bar.get_height()
        va = "bottom" if h >= 0 else "top"
        offset = 2 if h >= 0 else -2
        ax.annotate(f"{h:.3f}", (bar.get_x() + bar.get_width()/2, h),
                    xytext=(0, offset), textcoords="offset points", ha="center",
                    va=va, fontsize=7.5)
ax.axhline(0, color="black", linewidth=0.8, linestyle="-")
ax.set_xticks(x); ax.set_xticklabels(splits)
ax.set_ylabel("Forecast skill score\n(vs. 24h persistence)")
ax.set_title("(b) Forecast skill score", fontsize=9)
ax.spines[["top", "right"]].set_visible(False)
ax.set_ylim(min(skill_scratch) - 0.13, max(skill_transfer) * 1.18)
ax.text(0.98, 0.04, "below 0 = worse than persistence",
        ha="right", va="bottom", fontsize=6.5, style="italic", color="#555555",
        transform=ax.transAxes)

handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc="upper center", ncol=2, bbox_to_anchor=(0.5, 1.04),
           frameon=False, fontsize=8.5)

fig.tight_layout(rect=[0, 0, 1, 0.93])
fig.savefig("figure1_point_forecast_results.png", dpi=300, bbox_inches="tight")
fig.savefig("figure1_point_forecast_results.pdf", bbox_inches="tight")
print("Saved figure1_point_forecast_results.png and .pdf")

Saved figure1_point_forecast_results.png and .pdf


In [ ]:
"""
generate_figure2_hourly_coverage.py
====================================
Generates Figure 2 for the IConnect 2026 paper:
  "Bridging the Data Divide: Transfer Learning with Conformal Uncertainty
   Quantification for Solar PV Forecasting in Disruption-Prone Microgrids"

Figure: Hourly CQR coverage diagnostic for the 3-month models on the
        December 2023 clean test set (744 h), plotted alongside the
        training-data outage frequency at each hour of day.

Input CSV  : hourly_coverage_diagnostic.csv
Output PNG : figure2_hourly_coverage.png

Column reference
----------------
Unnamed: 0        -> hour of day (0-23); CSV is sorted by outage frequency,
                     so we re-sort by hour before plotting.
train_outage_freq -> fraction of training observations masked at that hour
                     under the BPDB load-shedding model.
scratch_coverage  -> empirical 90% CQR coverage, Scratch+CQR 3-month model.
transfer_coverage -> empirical 90% CQR coverage, Transfer+CQR 3-month model.
scratch_deficit   -> scratch_coverage - 0.90 (negative = over-covered).
transfer_deficit  -> transfer_coverage - 0.90 (negative = over-covered).

Usage
-----
    python generate_figure2_hourly_coverage.py

Dependencies: pandas, matplotlib, numpy
"""

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt

matplotlib.rcParams.update({
    "font.family": "DejaVu Sans",
    "axes.spines.top": False,
    "axes.spines.right": False,
})

# ── 1. Load and prepare data ──────────────────────────────────────────────────

df = pd.read_csv("hourly_coverage_diagnostic.csv")


# The first column is unnamed (hour index) — rename it
df = df.rename(columns={"Unnamed: 0": "hour"})

# The CSV is sorted by outage frequency descending; re-sort by hour (0-23)
df = df.sort_values("hour").reset_index(drop=True)

# Convert fractional values to percentages for plotting
df["scratch_pct"]  = df["scratch_coverage"]  * 100
df["transfer_pct"] = df["transfer_coverage"] * 100
df["outage_pct"]   = df["train_outage_freq"] * 100

# Keep daylight + shoulder hours only (05:00-22:00)
# Night hours are all 100% coverage (trivially) and add clutter
df_day = df[df["hour"].between(5, 22)].reset_index(drop=True)
hours  = df_day["hour"].values
x      = np.arange(len(hours))

# ── 2. Colour palette ─────────────────────────────────────────────────────────

COL_SCRATCH   = "#6b6b6b"   # neutral grey  -> Scratch+CQR
COL_TRANSFER  = "#1a5ea8"   # IEEE blue     -> Transfer+CQR
COL_TARGET    = "#c0392b"   # red dashed    -> 90% nominal line
COL_OUTAGE    = "#e67e22"   # orange        -> outage frequency line
BAR_WIDTH     = 0.32

# ── 3. Build figure ───────────────────────────────────────────────────────────

fig, ax1 = plt.subplots(figsize=(11, 4.8))
fig.patch.set_facecolor("white")
ax1.set_facecolor("white")

# --- Grouped bar chart: Scratch vs Transfer coverage -------------------------
ax1.bar(
    x - BAR_WIDTH / 2,
    df_day["scratch_pct"],
    BAR_WIDTH,
    color=COL_SCRATCH, alpha=0.85,
    label="Scratch+CQR  (3-month)",
    zorder=3,
)
ax1.bar(
    x + BAR_WIDTH / 2,
    df_day["transfer_pct"],
    BAR_WIDTH,
    color=COL_TRANSFER, alpha=0.85,
    label="Transfer+CQR (3-month)",
    zorder=3,
)

# --- Horizontal 90% nominal coverage line ------------------------------------
ax1.axhline(
    y=90, color=COL_TARGET, linewidth=1.8, linestyle="--",
    zorder=4, label="90% nominal target"
)

# --- Shade the problematic morning ramp-up window (06:00-09:00) --------------
morning_idx = np.where(np.isin(hours, [6, 7, 8, 9]))[0]
ax1.axvspan(
    morning_idx[0] - 0.55,
    morning_idx[-1] + 0.55,
    alpha=0.07, color=COL_TARGET, zorder=0
)
ax1.text(
    morning_idx.mean(), 109,
    "Morning ramp-up\n(06:00-09:00)",
    ha="center", va="center",
    fontsize=7.8, color=COL_TARGET, style="italic"
)

# --- Annotate the hour-6 Transfer anomaly (0% coverage) ---------------------
# --- Annotate the hour-6 Transfer anomaly (0% coverage) ---------------------
h6_xpos = np.where(hours == 6)[0][0]

ax1.annotate(
    "Transfer: 0% \n(Dawn mismatch)",
    xy=(h6_xpos + BAR_WIDTH / 2, 3),
    xytext=(h6_xpos + BAR_WIDTH / 2 + 2.6, 30),
    fontsize=8,
    color=COL_TARGET,
    ha="center",
    va="center",
    bbox=dict(
        boxstyle="round,pad=0.35",
        fc="white",          # box fill color
        ec=COL_TARGET,       # border color
        lw=1.2,
        alpha=0.95
    ),
    arrowprops=dict(
        arrowstyle="->",
        color=COL_TARGET,
        lw=1.2,
        shrinkA=4,
        shrinkB=2,
    ),
)

# --- Secondary axis: training outage frequency --------------------------------
ax2 = ax1.twinx()
ax2.plot(
    x, df_day["outage_pct"],
    color=COL_OUTAGE, linewidth=1.8, linestyle="-.",
    marker="D", markersize=3.8, alpha=0.80,
    label="Training outage freq. (%)",
    zorder=2,
)
ax2.set_ylabel(
    "Training outage frequency (%)",
    color=COL_OUTAGE, fontsize=10
)
ax2.tick_params(axis="y", labelcolor=COL_OUTAGE, labelsize=9)
ax2.set_ylim(0, 55)
ax2.spines["right"].set_visible(True)
ax2.spines["top"].set_visible(False)

# ── 4. Axis labels, ticks, grid ───────────────────────────────────────────────

ax1.set_xticks(x)
ax1.set_xticklabels(
    [f"{h:02d}:00" for h in hours],
    rotation=45, ha="right", fontsize=8.5
)
ax1.set_ylabel("Empirical CQR coverage (%)", fontsize=10)
ax1.set_xlabel(
    "Hour of day  —  December 2023 test set  "
    "(744 h total, daylight & shoulder hours shown)",
    fontsize=9.5, labelpad=6
)
ax1.set_ylim(0, 118)
ax1.set_yticks([0, 20, 40, 60, 80, 90, 100])
ax1.yaxis.grid(True, linestyle=":", alpha=0.45, zorder=0)
ax1.set_axisbelow(True)

# ── 5. Combined legend (both axes) ────────────────────────────────────────────

h1, l1 = ax1.get_legend_handles_labels()
h2, l2 = ax2.get_legend_handles_labels()
ax1.legend(
    h1 + h2, l1 + l2,
    loc="lower right", fontsize=8.8,
    framealpha=0.92, ncol=2,
    bbox_to_anchor=(1.0, 0.01)
)

# ── 6. Title and save ─────────────────────────────────────────────────────────

ax1.set_title(
    "Hourly Coverage Diagnostic — 3-month CQR Models vs. 90% Nominal Target",
    fontsize=10.8, fontweight="bold", pad=9
)

plt.tight_layout()
fig.savefig("figure2_hourly_coverage.png", dpi=200, bbox_inches="tight", facecolor="white")
plt.close()
print("Saved -> figure2_hourly_coverage.png")

# ── 7. Print a quick summary of below-target hours ────────────────────────────

print("\nHours below 90% nominal coverage:")
print(f"  {'Hour':>5}  {'Scratch%':>9}  {'Transfer%':>10}  {'OutageFreq%':>12}")
print("  " + "-" * 44)
for _, row in df.iterrows():
    if row["scratch_coverage"] < 0.90 or row["transfer_coverage"] < 0.90:
        print(
            f"  {int(row['hour']):02d}:00"
            f"  {row['scratch_pct']:>9.1f}"
            f"  {row['transfer_pct']:>10.1f}"
            f"  {row['outage_pct']:>12.1f}"
        )



Saved -> figure2_hourly_coverage.png

Hours below 90% nominal coverage:
   Hour   Scratch%   Transfer%   OutageFreq%
  --------------------------------------------
  06:00       86.7         0.0           9.9
  07:00       90.0        76.7          19.8
  08:00       86.7        70.0          23.1
  09:00       86.7        80.0          16.5


In [ ]:
"""
generate_figure3_hparam_search.py
===================================
Generates Figure 3 for the IConnect 2026 paper:
  "Bridging the Data Divide: Transfer Learning with Conformal Uncertainty
   Quantification for Solar PV Forecasting in Disruption-Prone Microgrids"

Figure: Boxplot distribution of validation MSE across all hyperparameter
        configurations per model tag, with the best (minimum) configuration
        in each tag highlighted by a red star.

Input CSV  : training_results.csv
Output PNG : figure3_hparam_search.png

Column reference
----------------
hidden_size  -> LSTM hidden units (32 or 64)
num_layers   -> number of stacked LSTM layers (1 or 2)
dropout      -> dropout probability (0.1 or 0.2)
lr           -> Adam learning rate
tag          -> model identifier:
                  PRETRAIN_DKASC  - source-domain pretraining (16 configs)
                  SCRATCH_1mo     - from-scratch, 1-month budget (16 configs)
                  TRANSFER_1mo    - fine-tuned, 1-month budget  (3 configs)
                  SCRATCH_3mo     - from-scratch, 3-month budget (16 configs)
                  TRANSFER_3mo    - fine-tuned, 3-month budget  (3 configs)
val_mse      -> validation MSE in kW^2 (lower is better)

Note: Transfer tags have only 3 lr values tested because the backbone
      architecture (hs=32, nl=2, dp=0.2) was fixed from pretraining.

Usage
-----
    python generate_figure3_hparam_search.py

Dependencies: pandas, matplotlib, numpy
"""

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

matplotlib.rcParams.update({
    "font.family": "DejaVu Sans",
    "axes.spines.top": False,
    "axes.spines.right": False,
})

# ── 1. Load data ──────────────────────────────────────────────────────────────

df = pd.read_csv("training_results.csv")

# Display order for the x-axis (left to right: pretrain -> scratch -> transfer)
TAG_ORDER = [
    "PRETRAIN_DKASC",
    "SCRATCH_1mo",
    "TRANSFER_1mo",
    "SCRATCH_3mo",
    "TRANSFER_3mo",
]

# Human-readable labels for x-axis ticks
XLABELS = [
    "Pretrain\n(DKASC)",
    "Scratch\n1-month",
    "Transfer\n1-month",
    "Scratch\n3-month",
    "Transfer\n3-month",
]

# Colour coding: blue shades for DKASC/transfer, grey for scratch
BOX_COLORS = [
    "#2c7bb6",   # PRETRAIN_DKASC  -> blue (source)
    "#6b6b6b",   # SCRATCH_1mo     -> grey
    "#1a5ea8",   # TRANSFER_1mo    -> dark blue
    "#9a9a9a",   # SCRATCH_3mo     -> slightly lighter grey
    "#1a5ea8",   # TRANSFER_3mo    -> dark blue
]

# ── 2. Prepare per-tag data ───────────────────────────────────────────────────

# Some low-lr configs produced very large val_mse (model did not converge).
# We clip at 2.5 kW^2 so that a single outlier does not collapse the y-axis.
# The clip value is noted on the axis label for transparency.
CLIP_VAL = 2.5

plot_data  = []   # list of arrays (one per tag), clipped
best_vals  = []   # minimum val_mse per tag (unclipped)
best_cfgs  = []   # best config dict per tag

for tag in TAG_ORDER:
    sub = df[df["tag"] == tag].copy()
    best_row = sub.loc[sub["val_mse"].idxmin()]
    best_vals.append(best_row["val_mse"])
    best_cfgs.append(best_row)

    clipped = np.clip(sub["val_mse"].values, 0, CLIP_VAL)
    plot_data.append(clipped)

# ── 3. Build figure ───────────────────────────────────────────────────────────

fig, ax = plt.subplots(figsize=(10, 4.6))
fig.patch.set_facecolor("white")
ax.set_facecolor("white")

x_positions = range(1, len(TAG_ORDER) + 1)

bp = ax.boxplot(
    plot_data,
    positions=list(x_positions),
    patch_artist=True,          # fill boxes with colour
    widths=0.50,
    notch=False,
    medianprops=dict(color="white", linewidth=2.5),
    whiskerprops=dict(linewidth=1.1, color="#444444"),
    capprops=dict(linewidth=1.1, color="#444444"),
    flierprops=dict(
        marker="x", markersize=5.5,
        markeredgewidth=1.0, alpha=0.55, color="#888888"
    ),
    zorder=3,
)

# Apply per-tag fill colours
for patch, col in zip(bp["boxes"], BOX_COLORS):
    patch.set_facecolor(col)
    patch.set_alpha(0.80)

# ── 4. Overlay best-config stars and value labels ─────────────────────────────

for i, (pos, bv) in enumerate(zip(x_positions, best_vals)):
    # Star marker at the best (minimum) val_mse value (clipped for display)
    star_y = min(bv, CLIP_VAL)
    ax.scatter(
        pos, star_y,
        color="#e74c3c", s=90, zorder=5,
        marker="*",
        label="Best config" if i == 0 else "",
    )
    # Numeric label just below the star
    offset = 0.055 if star_y > 0.20 else -0.055
    ax.text(
        pos, star_y - offset,
        f"{bv:.4f}",
        ha="center", va="top",
        fontsize=7.5, color="#c0392b", fontweight="bold"
    )

# ── 5. Vertical separator between pretrain / 1-month / 3-month groups ─────────

# Light dashed lines between the three logical groups
for xsep in [1.5, 3.5]:
    ax.axvline(xsep, color="#cccccc", linewidth=1.0, linestyle="--", zorder=0)

# Group label annotations
ax.text(1,   -0.22, "Source\npretraining", ha="center",
        fontsize=8, color="#555", transform=ax.get_xaxis_transform())
ax.text(2.5, -0.22, "1-month data budget",  ha="center",
        fontsize=8, color="#555", transform=ax.get_xaxis_transform())
ax.text(4.5, -0.22, "3-month data budget",  ha="center",
        fontsize=8, color="#555", transform=ax.get_xaxis_transform())

# ── 6. Axis labels, ticks, grid ───────────────────────────────────────────────

ax.set_xticks(list(x_positions))
ax.set_xticklabels(XLABELS, fontsize=9.8)
ax.set_ylabel(f"Validation MSE  (kW², clipped at {CLIP_VAL})", fontsize=10)
ax.set_xlabel("Model tag", fontsize=10, labelpad=4)
ax.yaxis.grid(True, linestyle=":", alpha=0.45, zorder=0)
ax.set_axisbelow(True)

# Y-axis: start slightly below 0 and end above the clip value
ax.set_ylim(-0.05, CLIP_VAL + 0.15)

# ── 7. Legend with colour patches ─────────────────────────────────────────────

legend_patches = [
    mpatches.Patch(color="#2c7bb6", alpha=0.80, label="DKASC pretrain"),
    mpatches.Patch(color="#6b6b6b", alpha=0.80, label="Scratch (random init)"),
    mpatches.Patch(color="#1a5ea8", alpha=0.80, label="Transfer (pretrained init)"),
    plt.Line2D([0], [0], marker="*", color="#e74c3c",
               linestyle="None", markersize=10, label="Best configuration"),
]
ax.legend(
    handles=legend_patches,
    loc="upper right", fontsize=8.8,
    framealpha=0.92, ncol=2
)

# ── 8. Title and save ─────────────────────────────────────────────────────────

ax.set_title(
    "Hyperparameter Search — Validation MSE Distribution per Model Tag",
    fontsize=10.8, fontweight="bold", pad=9
)

plt.tight_layout()
fig.savefig("figure3_hparam_search.png", dpi=200, bbox_inches="tight", facecolor="white")
plt.close()
print("Saved -> figure3_hparam_search.png")

# ── 9. Print best configurations table to console ─────────────────────────────

print("\nBest hyperparameter configuration per model tag:")
print(f"  {'Tag':<18}  {'h':>4}  {'L':>2}  {'drop':>5}  {'lr':>8}  {'val_mse':>9}")
print("  " + "-" * 56)
for tag, row in zip(TAG_ORDER, best_cfgs):
    print(
        f"  {tag:<18}"
        f"  {int(row['hidden_size']):>4}"
        f"  {int(row['num_layers']):>2}"
        f"  {row['dropout']:>5.2f}"
        f"  {row['lr']:>8.5f}"
        f"  {row['val_mse']:>9.4f}"
    )


Saved -> figure3_hparam_search.png

Best hyperparameter configuration per model tag:
  Tag                    h   L   drop        lr    val_mse
  --------------------------------------------------------
  PRETRAIN_DKASC        32   2   0.20   0.00100     0.1226
  SCRATCH_1mo           64   2   0.10   0.00100     0.6252
  TRANSFER_1mo          32   2   0.20   0.00010     0.5751
  SCRATCH_3mo           64   2   0.20   0.00100     0.5090
  TRANSFER_3mo          32   2   0.20   0.00050     0.4504


In [ ]:
"""
generate_pipeline_diagram.py
==============================
Generates the research methodology pipeline diagram for the IConnect 2026 paper:
  "Bridging the Data Divide: Transfer Learning with Conformal Uncertainty
   Quantification for Solar PV Forecasting in Disruption-Prone Microgrids"

The diagram shows the complete workflow:
  Source domain  : DKASC raw data -> cleaning -> LSTM pretraining
  Target domain  : NASA POWER -> pvlib simulation -> BPDB masking -> splits
  Model variants : Scratch (1-mo, 3-mo) | Transfer (1-mo, 3-mo)
  Post-hoc       : CQR calibration
  Evaluation     : Clean test set -> point & probabilistic metrics

Output PNG : figure0_pipeline_diagram.png

Dependencies: matplotlib, numpy
"""

import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch

matplotlib.rcParams.update({"font.family": "DejaVu Sans"})

# ── Colour palette ─────────────────────────────────────────────────────────────
C = {
    "purple"  : {"face": "#6A0DAD", "edge": "#4B0082", "text": "white"},  # data sources
    "teal"    : {"face": "#0D7377", "edge": "#095C5F", "text": "white"},  # processed data
    "blue"    : {"face": "#1a5ea8", "edge": "#13467F", "text": "white"},  # pretrain / transfer
    "gray"    : {"face": "#5a5a5a", "edge": "#3d3d3d", "text": "white"},  # scratch models
    "orange"  : {"face": "#D97706", "edge": "#B45309", "text": "white"},  # CQR / calibration
    "green"   : {"face": "#166534", "edge": "#14532D", "text": "white"},  # evaluation
    "neutral" : {"face": "#E5E7EB", "edge": "#9CA3AF", "text": "#1f2937"}, # splits
}

ARROW_KW = dict(
    arrowstyle="-|>",
    color="#374151",
    lw=1.3,
    mutation_scale=12,
    zorder=5,
)
DASH_ARROW_KW = dict(
    arrowstyle="-|>",
    color="#1a5ea8",
    lw=1.2,
    linestyle="dashed",
    mutation_scale=11,
    connectionstyle="arc3,rad=0.0",
    zorder=5,
)

# ── Canvas ──────────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(16, 11))
ax.set_xlim(0, 16)
ax.set_ylim(0, 11)
ax.axis("off")
fig.patch.set_facecolor("white")

# ── Helper functions ────────────────────────────────────────────────────────────

def draw_box(ax, cx, cy, w, h, title, subtitle=None,
             color_key="teal", fontsize_title=9.0, fontsize_sub=7.5, radius=0.18):
    """Draw a rounded box centred at (cx, cy) with optional subtitle."""
    col = C[color_key]
    box = FancyBboxPatch(
        (cx - w / 2, cy - h / 2), w, h,
        boxstyle=f"round,pad=0,rounding_size={radius}",
        facecolor=col["face"], edgecolor=col["edge"],
        linewidth=1.2, zorder=3,
    )
    ax.add_patch(box)

    if subtitle:
        ax.text(cx, cy + h * 0.13, title,
                ha="center", va="center",
                color=col["text"], fontsize=fontsize_title,
                fontweight="bold", zorder=4)
        ax.text(cx, cy - h * 0.22, subtitle,
                ha="center", va="center",
                color=col["text"], fontsize=fontsize_sub,
                alpha=0.92, zorder=4)
    else:
        ax.text(cx, cy, title,
                ha="center", va="center",
                color=col["text"], fontsize=fontsize_title,
                fontweight="bold", zorder=4)

    return (cx - w/2, cy - h/2, cx + w/2, cy + h/2)   # (x0,y0,x1,y1)


def arrow(ax, x0, y0, x1, y1, dashed=False, rad=0.0, color=None):
    """Draw a simple directional arrow between two points."""
    kw = DASH_ARROW_KW.copy() if dashed else ARROW_KW.copy()
    if color:
        kw["color"] = color
    if rad != 0.0:
        kw["connectionstyle"] = f"arc3,rad={rad}"
    patch = FancyArrowPatch((x0, y0), (x1, y1), **kw)
    ax.add_patch(patch)


def section_label(ax, cx, cy, text, color="#374151"):
    """Draw a small italic section label above a group of boxes."""
    ax.text(cx, cy, text,
            ha="center", va="center",
            color=color, fontsize=8.0, style="italic",
            fontweight="bold", zorder=4)


def hline(ax, x0, x1, y, color="#9CA3AF", lw=0.9, ls="--"):
    ax.plot([x0, x1], [y, y], color=color, lw=lw, ls=ls, zorder=2)

# ══════════════════════════════════════════════════════════════════════════════
#  ROW LAYOUT  (y coordinates, top -> bottom)
#  R1 = 9.8   Data sources
#  R2 = 8.3   Preprocessing
#  R3 = 6.8   Pretrain backbone  |  Bangladesh masked splits
#  R4 = 5.2   Four model variants
#  R5 = 3.7   CQR calibration  (wide)
#  R6 = 2.4   Clean test set
#  R7 = 1.0   Evaluation metrics
# ══════════════════════════════════════════════════════════════════════════════

ROW = {"R1": 9.8, "R2": 8.3, "R3": 6.8, "R4": 5.2,
       "R5": 3.7, "R6": 2.4, "R7": 1.0}

BH  = 0.82   # standard box height
BH2 = 0.65   # shorter box height (splits, eval)
BW  = 3.0    # wide box
BWs = 2.3    # slim box (model variants)

# ── Section annotations ─────────────────────────────────────────────────────
section_label(ax, 3.5, 10.55, "SOURCE DOMAIN  (DKASC Alice Springs, Australia)", "#6A0DAD")
section_label(ax, 12.0, 10.55, "TARGET DOMAIN  (Bangladesh, 23.8°N / 90.4°E)", "#0D7377")

# ── ROW 1 : Data Sources ─────────────────────────────────────────────────────
draw_box(ax, 3.5, ROW["R1"], BW, BH,
         "DKASC Alice Springs",
         "Raw 5-min PV measurements\n(Site 106-C, 2008–2023)",
         color_key="purple")

draw_box(ax, 12.0, ROW["R1"], BW, BH,
         "NASA POWER Reanalysis",
         "Hourly meteorological fields\n(GHI, temperature, 0.5° grid)",
         color_key="purple")

# ── ROW 2 : Preprocessing ────────────────────────────────────────────────────
draw_box(ax, 3.5, ROW["R2"], BW, BH,
         "Data Cleaning & Resampling",
         "148,537 valid hourly observations\nfeatures: GHI, Temp, sin/cos hour",
         color_key="teal")

draw_box(ax, 10.0, ROW["R2"], 2.6, BH,
         "pvlib PV Simulation",
         "10 kWp rooftop · PVWATTS\nPerez transposition · Ineichen sky",
         color_key="teal")

draw_box(ax, 13.6, ROW["R2"], 2.6, BH,
         "BPDB Outage Masking",
         "Seasonal outage probability\napplied as Bernoulli mask",
         color_key="neutral", fontsize_title=8.8)

# ── ROW 3 : Pretrain + Masked Splits ─────────────────────────────────────────
draw_box(ax, 3.5, ROW["R3"], BW, BH,
         "LSTM Pretraining",
         "16 hyperparameter configs · DKASC source\nBest: h=32, L=2, p=0.2, lr=1e-3  →  val MSE=0.123",
         color_key="blue")

draw_box(ax, 10.0, ROW["R3"], 2.6, BH2,
         "1-month Masked Split",
         "November 2023 · 720 h\n(BPDB mask applied)",
         color_key="neutral", fontsize_title=8.8)

draw_box(ax, 13.6, ROW["R3"], 2.6, BH2,
         "3-month Masked Split",
         "Sep–Nov 2023 · 2,160 h\n(BPDB mask applied)",
         color_key="neutral", fontsize_title=8.8)

# ── ROW 4 : Four Model Variants ──────────────────────────────────────────────
CX = [2.0, 5.0, 9.3, 13.5]   # x-centres of the four model boxes

draw_box(ax, CX[0], ROW["R4"], BWs, BH,
         "Scratch  |  1-month",
         "Random init · 16-config search\nBest: h=64, L=2, p=0.1, lr=1e-3",
         color_key="gray")

draw_box(ax, CX[1], ROW["R4"], BWs, BH,
         "Transfer  |  1-month",
         "Pretrained init · fine-tune\nBest: h=32, L=2, p=0.2, lr=1e-4",
         color_key="blue")

draw_box(ax, CX[2], ROW["R4"], BWs, BH,
         "Scratch  |  3-month",
         "Random init · 16-config search\nBest: h=64, L=2, p=0.2, lr=1e-3",
         color_key="gray")

draw_box(ax, CX[3], ROW["R4"], BWs, BH,
         "Transfer  |  3-month",
         "Pretrained init · fine-tune\nBest: h=32, L=2, p=0.2, lr=5e-4",
         color_key="blue")

# model pair labels
ax.text(3.5, ROW["R4"] + BH/2 + 0.18, "1-month data budget",
        ha="center", fontsize=8, color="#374151", style="italic")
ax.text(11.4, ROW["R4"] + BH/2 + 0.18, "3-month data budget",
        ha="center", fontsize=8, color="#374151", style="italic")

# ── ROW 5 : CQR Calibration ──────────────────────────────────────────────────
draw_box(ax, 7.75, ROW["R5"], 14.2, BH2,
         "Conformal Quantile Regression  (CQR)",
         "Calibration set (20% of masked training data) · nonconformity score · q̂ · "
         "90% nominal coverage target\n"
         "C(x) = [q̂ˡᵒ(x) − q̂,  q̂ʰⁱ(x) + q̂]",
         color_key="orange")

# ── ROW 6 : Clean Test Set ───────────────────────────────────────────────────
draw_box(ax, 7.75, ROW["R6"], 5.5, BH2,
         "Clean Test Set",
         "December 2023 · 744 hourly observations (no missing values)",
         color_key="teal")

# ── ROW 7 : Evaluation Metrics ───────────────────────────────────────────────
draw_box(ax, 4.2, ROW["R7"], 6.5, BH2,
         "Point-Forecast Metrics",
         "RMSE  ·  MAE  ·  Skill score vs. 24 h persistence",
         color_key="green")

draw_box(ax, 11.8, ROW["R7"], 6.5, BH2,
         "Probabilistic Metrics",
         "Coverage  ·  Interval width  ·  Pinball loss  ·  Hourly diagnostic",
         color_key="green")

# ══════════════════════════════════════════════════════════════════════════════
#  ARROWS
# ══════════════════════════════════════════════════════════════════════════════

# ── Source domain vertical flow ───────────────────────────────────────────────
# DKASC raw -> clean
arrow(ax, 3.5, ROW["R1"] - BH/2,  3.5, ROW["R2"] + BH/2)
# DKASC clean -> pretrain
arrow(ax, 3.5, ROW["R2"] - BH/2,  3.5, ROW["R3"] + BH/2)

# ── Target domain vertical flow ───────────────────────────────────────────────
# NASA POWER -> pvlib sim
arrow(ax, 12.0, ROW["R1"] - BH/2,  10.0, ROW["R2"] + BH/2)
# pvlib sim -> BPDB masking
arrow(ax, 11.3, ROW["R2"],  12.9, ROW["R2"])
# BPDB masking -> 1-month split
arrow(ax, 12.6, ROW["R2"] - BH/2,  10.0, ROW["R3"] + BH2/2)
# BPDB masking -> 3-month split
arrow(ax, 13.6, ROW["R2"] - BH/2,  13.6, ROW["R3"] + BH2/2)

# ── Splits -> model variants (solid grey — training data) ─────────────────────
# 1-month split -> Scratch 1mo
arrow(ax, 9.2, ROW["R3"] - BH2/2,   CX[0], ROW["R4"] + BH/2,  rad=0.15)
# 1-month split -> Transfer 1mo
arrow(ax, 10.8, ROW["R3"] - BH2/2,  CX[1], ROW["R4"] + BH/2,  rad=-0.12)
# 3-month split -> Scratch 3mo
arrow(ax, 12.6, ROW["R3"] - BH2/2,  CX[2], ROW["R4"] + BH/2,  rad=0.12)
# 3-month split -> Transfer 3mo
arrow(ax, 14.5, ROW["R3"] - BH2/2,  CX[3], ROW["R4"] + BH/2,  rad=-0.10)

# ── Pretrained backbone -> Transfer models (dashed blue) ─────────────────────
# Pretrain -> Transfer 1mo
arrow(ax, 5.0, ROW["R3"],   CX[1], ROW["R4"] + BH/2,
      dashed=True, rad=-0.25, color="#1a5ea8")
# Pretrain -> Transfer 3mo
arrow(ax, 5.0, ROW["R3"],   CX[3], ROW["R4"] + BH/2,
      dashed=True, rad=-0.35, color="#1a5ea8")

# Backbone weight label mid-arc (approximate midpoint)
ax.text(7.5, ROW["R3"] - 0.45,
        "LSTM backbone\nweights",
        ha="center", va="center",
        fontsize=7.2, color="#1a5ea8", style="italic",
        bbox=dict(facecolor="white", edgecolor="#1a5ea8",
                  linewidth=0.7, boxstyle="round,pad=0.2", alpha=0.85))

# ── Model variants -> CQR ─────────────────────────────────────────────────────
for cx in CX:
    arrow(ax, cx, ROW["R4"] - BH/2,  cx, ROW["R5"] + BH2/2)

# ── CQR -> Test set ───────────────────────────────────────────────────────────
arrow(ax, 7.75, ROW["R5"] - BH2/2,  7.75, ROW["R6"] + BH2/2)

# ── Test set -> Evaluation (Y-branch) ────────────────────────────────────────
# centre trunk
ax.plot([7.75, 7.75], [ROW["R6"] - BH2/2, 1.55],
        color="#374151", lw=1.3, zorder=5)
# horizontal
ax.plot([4.2, 11.8], [1.55, 1.55],
        color="#374151", lw=1.3, zorder=5)
# down to left eval box
arrow(ax, 4.2, 1.55,  4.2, ROW["R7"] + BH2/2)
# down to right eval box
arrow(ax, 11.8, 1.55,  11.8, ROW["R7"] + BH2/2)

# ══════════════════════════════════════════════════════════════════════════════
#  LEGEND
# ══════════════════════════════════════════════════════════════════════════════

legend_items = [
    mpatches.Patch(facecolor=C["purple"]["face"],  edgecolor=C["purple"]["edge"],
                   label="External data source"),
    mpatches.Patch(facecolor=C["teal"]["face"],    edgecolor=C["teal"]["edge"],
                   label="Processed / simulated data"),
    mpatches.Patch(facecolor=C["blue"]["face"],    edgecolor=C["blue"]["edge"],
                   label="Pretrain / Transfer model"),
    mpatches.Patch(facecolor=C["gray"]["face"],    edgecolor=C["gray"]["edge"],
                   label="Scratch model"),
    mpatches.Patch(facecolor=C["neutral"]["face"], edgecolor=C["neutral"]["edge"],
                   label="Masked training split"),
    mpatches.Patch(facecolor=C["orange"]["face"],  edgecolor=C["orange"]["edge"],
                   label="CQR calibration"),
    mpatches.Patch(facecolor=C["green"]["face"],   edgecolor=C["green"]["edge"],
                   label="Evaluation"),
    plt.Line2D([0], [0], color="#374151", lw=1.3, label="Data / training flow"),
    plt.Line2D([0], [0], color="#1a5ea8", lw=1.2, ls="--",
               label="Backbone weight transfer"),
]

ax.legend(
    handles=legend_items,
    loc="lower left",
    bbox_to_anchor=(0.0, -0.05),
    ncol=1,
    fontsize=8.2,
    framealpha=1.0,              # fully opaque
    facecolor="white",           # pure white legend
    edgecolor="#D1D5DB",
    title="Legend",
    title_fontsize=8.5,
)

# ── Title ─────────────────────────────────────────────────────────────────────
ax.set_title(
    "Research Methodology Pipeline\n"
    "Transfer Learning + Conformal Quantile Regression for Solar PV Forecasting "
    "in Data-Scarce, Load-Shedding-Affected Microgrids",
    fontsize=11.5, fontweight="bold", pad=12, color="#111827"
)

plt.tight_layout()
fig.savefig("figure0_pipeline_diagram.png",
            dpi=200, bbox_inches="tight", facecolor="white")
plt.close()
print("Saved -> figure0_pipeline_diagram.png")


Saved -> figure0_pipeline_diagram.png
